# Modeling and Betting Backtest

## NFL Spread Prediction Using Machine Learning

This notebook uses the final EDA dataset to build and evaluate NFL point-margin prediction models and then converts those predictions into simulated spread bets.

The target is `home_margin`, defined as:

`home_score - away_score`

The modeling workflow has two related but distinct goals:

1. **Prediction:** estimate the final home-team margin as accurately as possible.
2. **Betting:** compare the model prediction with `market_home_margin` and place a simulated bet only when the disagreement exceeds a selected threshold.

The analysis begins with **4,175 regular-season games from 2010–2025**. Week 1 is removed because the current-season rolling features do not yet have prior observations, leaving **3,920 games** for modeling. The notebook then compares several regression algorithms, tests Random Forest-based feature selection, evaluates the final holdout seasons, and performs a walk-forward historical backtest.

Betting performance is measured using actual spread prices and a fixed **$100 risk per bet**. Results are evaluated with MAE, RMSE, R², win rate, profit, ROI, sample size, season consistency, and situational performance.

In [1]:
# Core data analysis packages
import pandas as pd
import numpy as np
from pathlib import Path

# Visualization
import plotly.express as px
import matplotlib.pyplot as plt

# Machine learning preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Regression models for predicting score margin
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

# Classification model for optional home-cover prediction
from sklearn.naive_bayes import GaussianNB

# Model evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.3f}".format)

print("Packages imported successfully.")

Packages imported successfully.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path

# Persistent project folder in Google Drive
project_root = Path("/content/drive/MyDrive/DATA 606/nfl_spread_capstone")

data_dir = project_root / "data"
raw_dir = data_dir / "raw"
processed_dir = data_dir / "processed"
notebooks_dir = project_root / "notebooks"
docs_dir = project_root / "docs"
models_dir = project_root / "models"

# Create folders if they do not already exist
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
notebooks_dir.mkdir(parents=True, exist_ok=True)
docs_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Raw data folder:", raw_dir)
print("Processed data folder:", processed_dir)
print("Notebooks folder:", notebooks_dir)
print("Docs folder:", docs_dir)
print("Models folder:", models_dir)

Project root: /content/drive/MyDrive/DATA 606/nfl_spread_capstone
Raw data folder: /content/drive/MyDrive/DATA 606/nfl_spread_capstone/data/raw
Processed data folder: /content/drive/MyDrive/DATA 606/nfl_spread_capstone/data/processed
Notebooks folder: /content/drive/MyDrive/DATA 606/nfl_spread_capstone/notebooks
Docs folder: /content/drive/MyDrive/DATA 606/nfl_spread_capstone/docs
Models folder: /content/drive/MyDrive/DATA 606/nfl_spread_capstone/models


In [4]:
print("Processed files available:")

for file in processed_dir.iterdir():
    print(file.name)

Processed files available:
nfl_spread_eda_dataset_2010_2025.gsheet
nfl_spread_eda_dataset_2010_2025.csv
nflverse_games_cleaned_2010_2025.csv


## Load Final EDA and Game-Level Data

The notebook loads the two processed files created earlier in the project:

- `nfl_spread_eda_dataset_2010_2025.csv`: **4,175 rows × 212 columns**, containing the target, market margin, game context, and engineered pregame team features.
- `nflverse_games_cleaned_2010_2025.csv`: **4,175 rows × 52 columns**, used primarily to supply the home and away spread prices needed for profit calculations.

The spread-odds merge preserves the original **4,175-game row count**, confirming that no duplicate games are introduced. The required modeling columns are present in both sources where expected.

In [5]:
# Load final EDA dataset
eda_file = processed_dir / "nfl_spread_eda_dataset_2010_2025.csv"
df = pd.read_csv(eda_file)

print("EDA dataset shape:", df.shape)
df.head()

EDA dataset shape: (4175, 212)


,game_id,season,week,gameday,weekday,away_team,home_team,away_score,home_score,home_margin,home_spread,market_home_margin,spread_result,home_cover,push,total_line,home_rest,away_rest,roof,surface,temp,wind,div_game,stadium,home_moneyline,home_spread_odds,home_qb_id,home_qb_name,home_coach,home_offensive_plays,home_off_epa_per_play,home_off_success_rate,home_yards_per_play,home_pass_rate,home_pass_epa_per_play,home_pass_success_rate,home_yards_per_pass,home_avg_air_yards,home_avg_yac,home_avg_cpoe,home_avg_xpass,home_avg_pass_oe,home_rush_rate,home_rush_epa_per_play,home_rush_success_rate,home_yards_per_rush,home_early_down_pass_rate,home_early_down_pass_success_rate,home_early_down_epa_per_play,home_early_down_pass_epa,home_pressure_allowed_rate,home_sack_rate,home_qb_hit_rate_allowed,home_negative_play_rate,home_turnover_rate,home_explosive_play_rate,home_explosive_pass_rate,home_explosive_rush_rate,home_third_down_attempts,home_third_down_conversions,home_fourth_down_attempts,home_fourth_down_conversions,home_avg_xyac_epa,home_avg_xyac_yards,home_avg_xyac_success,home_third_down_conversion_rate,home_fourth_down_conversion_rate,home_defensive_plays,home_def_epa_allowed_per_play,home_def_success_rate_allowed,home_def_yards_allowed_per_play,home_def_pass_epa_allowed,home_def_pass_success_allowed,home_def_yards_allowed_per_pass,home_def_rush_epa_allowed,home_def_rush_success_allowed,home_def_yards_allowed_per_rush,home_def_early_down_epa_allowed,home_def_early_down_pass_epa_allowed,home_def_early_down_pass_success_allowed,home_pressure_created_rate,home_sack_created_rate,home_qb_hit_created_rate,home_tackle_for_loss_rate,home_explosive_allowed_rate,home_explosive_pass_allowed_rate,home_explosive_rush_allowed_rate,home_turnover_forced_rate,away_moneyline,away_spread_odds,away_qb_id,away_qb_name,away_coach,away_offensive_plays,away_off_epa_per_play,away_off_success_rate,away_yards_per_play,away_pass_rate,away_pass_epa_per_play,away_pass_success_rate,away_yards_per_pass,away_avg_air_yards,away_avg_yac,away_avg_cpoe,away_avg_xpass,away_avg_pass_oe,away_rush_rate,away_rush_epa_per_play,away_rush_success_rate,away_yards_per_rush,away_early_down_pass_rate,away_early_down_pass_success_rate,away_early_down_epa_per_play,away_early_down_pass_epa,away_pressure_allowed_rate,away_sack_rate,away_qb_hit_rate_allowed,away_negative_play_rate,away_turnover_rate,away_explosive_play_rate,away_explosive_pass_rate,away_explosive_rush_rate,away_third_down_attempts,away_third_down_conversions,away_fourth_down_attempts,away_fourth_down_conversions,away_avg_xyac_epa,away_avg_xyac_yards,away_avg_xyac_success,away_third_down_conversion_rate,away_fourth_down_conversion_rate,away_defensive_plays,away_def_epa_allowed_per_play,away_def_success_rate_allowed,away_def_yards_allowed_per_play,away_def_pass_epa_allowed,away_def_pass_success_allowed,away_def_yards_allowed_per_pass,away_def_rush_epa_allowed,away_def_rush_success_allowed,away_def_yards_allowed_per_rush,away_def_early_down_epa_allowed,away_def_early_down_pass_epa_allowed,away_def_early_down_pass_success_allowed,away_pressure_created_rate,away_sack_created_rate,away_qb_hit_created_rate,away_tackle_for_loss_rate,away_explosive_allowed_rate,away_explosive_pass_allowed_rate,away_explosive_rush_allowed_rate,away_turnover_forced_rate,offensive_plays_diff,off_epa_per_play_diff,off_success_rate_diff,yards_per_play_diff,pass_rate_diff,pass_epa_per_play_diff,pass_success_rate_diff,yards_per_pass_diff,avg_air_yards_diff,avg_yac_diff,avg_cpoe_diff,avg_xpass_diff,avg_pass_oe_diff,rush_rate_diff,rush_epa_per_play_diff,rush_success_rate_diff,yards_per_rush_diff,early_down_pass_rate_diff,early_down_pass_success_rate_diff,early_down_epa_per_play_diff,early_down_pass_epa_diff,pressure_allowed_rate_diff,sack_rate_diff,qb_hit_rate_allowed_diff,negative_play_rate_diff,turnover_rate_diff,explosive_play_rate_diff,explosive_pass_rate_diff,explosive_rush_rate_diff,third_down_attempts_diff,third_down_conversions_diff,fourth_down_

In [6]:
# Load cleaned game-level dataset
games_file = processed_dir / "nflverse_games_cleaned_2010_2025.csv"
games_cleaned = pd.read_csv(games_file)

print("Cleaned games dataset shape:", games_cleaned.shape)
games_cleaned.head()

Cleaned games dataset shape: (4175, 52)


,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,home_score,location,result,total,overtime,old_game_id,gsis,nfl_detail_id,pfr,pff,espn,ftn,away_rest,home_rest,away_moneyline,home_moneyline,spread_line,away_spread_odds,home_spread_odds,total_line,under_odds,over_odds,div_game,roof,surface,temp,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium,home_margin,market_home_margin,home_spread,spread_result,home_cover,push
0,2010_01_MIN_NO,2010,REG,1,2010-09-09,Thursday,20:30,MIN,9.000,NO,14.000,Home,5.000,23.000,0.000,2010090900,54863.000,NaN,201009090nor,1727.000,300909018,NaN,7,7,197.000,-220.000,4.500,-105.000,-103.000,48.500,-104.000,-106.000,0,dome,sportturf,NaN,NaN,00-0005106,00-0020531,Brett Favre,Drew Brees,Brad Childress,Sean Payton,Terry McAulay,NOR00,Louisiana Superdome,5.000,4.500,-4.500,0.500,1,0
1,2010_01_MIA_BUF,2010,REG,1,2010-09-12,Sunday,13:00,MIA,15.000,BUF,10.000,Home,-5.000,25.000,0.000,2010091201,54864.000,NaN,201009120buf,1729.000,300912002,NaN,7,7,-155.000,140.000,-3.000,-106.000,-102.000,39.500,-110.000,100.000,1,outdoors,astroplay,62.000,7.000,00-0026197,00-0025479,Chad Henne,Trent Edwards,Tony Sparano,Chan Gailey,Clete Blakeman,BUF00,Ralph Wilson Stadium,-5.000,-3.000,3.000,-2.000,0,0
2,2010_01_DET_CHI,2010,REG,1,2010-09-12,Sunday,13:00,DET,14.000,CHI,19.000,Home,5.000,33.000,0.000,2010091207,54865.000,NaN,201009120chi,1736.000,300912003,NaN,7,7,248.000,-280.000,6.500,103.000,-111.000,44.500,-105.000,-105.000,1,outdoors,grass,75.000,10.000,00-0026498,00-0024226,Matthew Stafford,Jay Cutler,Jim Schwartz,Lovie Smith,Gene Steratore,CHI98,Soldier Field,5.000,6.500,-6.500,-1.500,0,0
3,2010_01_IND_HOU,2010,REG,1,2010-09-12,Sunday,13:00,IND,24.000,HOU,34.000,Home,10.000,58.000,0.000,2010091203,54866.000,NaN,201009120htx,1731.000,300912034,NaN,7,7,-117.000,106.000,-1.000,-110.000,102.000,47.500,-102.000,-108.000,1,closed,grass,NaN,NaN,00-0010346,00-0022787,Peyton Manning,Matt Schaub,Jim Caldwell,Gary Kubiak,Ed Hochuli,HOU00,Reliant Stadium,10.000,-1.000,1.000,11.000,1,0
4,2010_01_DEN_JAX,2010,REG,1,2010-09-12,Sunday,13:00,DEN,17.000,JAX,24.000,Home,7.000,41.000,0.000,2010091204,54867.000,NaN,201009120jax,1732.000,300912030,NaN,7,7,166.000,-185.000,3.000,109.000,-118.000,41.500,-110.000,100.000,0,outdoors,grass,90.000,10.000,00-0023541,00-0021231,Kyle Orton,David Garrard,Josh McDaniels,Jack Del Rio,Walt Coleman,JAX00,EverBank Field,7.000,3.000,-3.000,4.000,1,0


In [7]:
print("EDA dataset columns:")
print(df.columns.tolist())

print("\nCleaned games dataset columns:")
print(games_cleaned.columns.tolist())

EDA dataset columns:
['game_id', 'season', 'week', 'gameday', 'weekday', 'away_team', 'home_team', 'away_score', 'home_score', 'home_margin', 'home_spread', 'market_home_margin', 'spread_result', 'home_cover', 'push', 'total_line', 'home_rest', 'away_rest', 'roof', 'surface', 'temp', 'wind', 'div_game', 'stadium', 'home_moneyline', 'home_spread_odds', 'home_qb_id', 'home_qb_name', 'home_coach', 'home_offensive_plays', 'home_off_epa_per_play', 'home_off_success_rate', 'home_yards_per_play', 'home_pass_rate', 'home_pass_epa_per_play', 'home_pass_success_rate', 'home_yards_per_pass', 'home_avg_air_yards', 'home_avg_yac', 'home_avg_cpoe', 'home_avg_xpass', 'home_avg_pass_oe', 'home_rush_rate', 'home_rush_epa_per_play', 'home_rush_success_rate', 'home_yards_per_rush', 'home_early_down_pass_rate', 'home_early_down_pass_success_rate', 'home_early_down_epa_per_play', 'home_early_down_pass_epa', 'home_pressure_allowed_rate', 'home_sack_rate', 'home_qb_hit_rate_allowed', 'home_negative_play_rate

In [8]:
required_model_cols = [
    "game_id",
    "season",
    "week",
    "home_team",
    "away_team",
    "home_margin",
    "market_home_margin"
]

required_odds_cols = [
    "game_id",
    "home_spread_odds",
    "away_spread_odds"
]

missing_model_cols = [col for col in required_model_cols if col not in df.columns]
missing_odds_cols = [col for col in required_odds_cols if col not in games_cleaned.columns]

print("Missing from EDA dataset:")
print(missing_model_cols)

print("\nMissing from cleaned games dataset:")
print(missing_odds_cols)

Missing from EDA dataset:
[]

Missing from cleaned games dataset:
[]


In [9]:
# Keep the odds columns from the cleaned games dataset
odds_cols = [
    "game_id",
    "home_spread_odds",
    "away_spread_odds",
    "home_moneyline",
    "away_moneyline"
]

odds_cols = [col for col in odds_cols if col in games_cleaned.columns]

# Only merge odds columns that are not already in df
odds_to_merge = [
    col for col in odds_cols
    if col != "game_id" and col not in df.columns
]

if len(odds_to_merge) > 0:
    df = df.merge(
        games_cleaned[["game_id"] + odds_to_merge],
        on="game_id",
        how="left"
    )

print("Dataset shape after odds merge:", df.shape)

df[
    [
        "game_id",
        "season",
        "week",
        "home_team",
        "away_team",
        "home_margin",
        "market_home_margin",
        "home_spread_odds",
        "away_spread_odds"
    ]
].head()

Dataset shape after odds merge: (4175, 212)


,game_id,season,week,home_team,away_team,home_margin,market_home_margin,home_spread_odds,away_spread_odds
0,2010_01_MIN_NO,2010,1,NO,MIN,5.000,4.500,-103.000,-105.000
1,2010_01_MIA_BUF,2010,1,BUF,MIA,-5.000,-3.000,-102.000,-106.000
2,2010_01_DET_CHI,2010,1,CHI,DET,5.000,6.500,-111.000,103.000
3,2010_01_IND_HOU,2010,1,HOU,IND,10.000,-1.000,102.000,-110.000
4,2010_01_DEN_JAX,2010,1,JAX,DEN,7.000,3.000,-118.000,109.000


In [10]:
# Confirm market expected home margin has a positive relationship with actual home margin
df[["market_home_margin", "home_margin"]].corr()

,market_home_margin,home_margin
market_home_margin,1.000,0.435
home_margin,0.435,1.000


## Spread Direction Check

Before modeling, the notebook verifies that the sportsbook spread is aligned with the target from the **home-team perspective**.

The observed correlation between `market_home_margin` and actual `home_margin` is **0.435**, which confirms the expected positive relationship.

Therefore:

- `home_margin` = actual final margin for the home team.
- `market_home_margin` = sportsbook expected home margin.
- Positive `model_edge` = the model favors the home side more than the market.
- Negative `model_edge` = the model favors the away side more than the market.

This sign check is important because an incorrect spread direction would reverse every later betting decision.

In [11]:
# Start with a copy of the full dataset
model_df = df.copy()

# Make sure week is numeric
model_df["week"] = pd.to_numeric(model_df["week"], errors="coerce")

# Check rows before filtering
print("Rows before filtering:", model_df.shape[0])

# Keep rows with valid target and market margin
model_df = model_df[
    model_df["home_margin"].notna() &
    model_df["market_home_margin"].notna()
].copy()

print("Rows after removing missing target/market margin:", model_df.shape[0])

# Drop Week 1 because pregame rolling team features are unavailable
model_df = model_df[model_df["week"] > 1].copy()

print("Rows after dropping Week 1:", model_df.shape[0])
print("Seasons:", model_df["season"].min(), "to", model_df["season"].max())
print("Weeks included:", sorted(model_df["week"].dropna().unique()))
print("Unique games:", model_df["game_id"].nunique())

Rows before filtering: 4175
Rows after removing missing target/market margin: 4175
Rows after dropping Week 1: 3920
Seasons: 2010 to 2025
Weeks included: [np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]
Unique games: 3920


In [12]:
# Check remaining missing values after dropping Week 1
missing_after_week1_drop = (
    model_df
    .isna()
    .sum()
    .reset_index()
)

missing_after_week1_drop.columns = ["column", "missing_count"]
missing_after_week1_drop["missing_percent"] = (
    missing_after_week1_drop["missing_count"] / len(model_df) * 100
)

missing_after_week1_drop = missing_after_week1_drop.sort_values(
    "missing_percent",
    ascending=False
)

missing_after_week1_drop.head(100)

,column,missing_count,missing_percent
21,wind,1248,31.837
20,temp,1248,31.837
189,fourth_down_conversion_rate_diff,293,7.474
130,away_fourth_down_conversion_rate,182,4.643
66,home_fourth_down_conversion_rate,159,4.056
211,surface_clean,26,0.663
19,surface,26,0.663
207,explosive_allowed_rate_diff,2,0.051
200,def_early_down_epa_allowed_diff,2,0.051
208,explosive_pass_allowed_rate_diff,2,0.051


### Model-Ready Dataset Check

After removing Week 1, the modeling table contains **3,920 games from 2010–2025**. Most engineered pregame variables are now nearly complete.

The main remaining missingness is concentrated in a few interpretable fields:

- `temp`: **1,248 missing (31.8%)**
- `wind`: **1,248 missing (31.8%)**
- `fourth_down_conversion_rate_diff`: **293 missing (7.5%)**
- away fourth-down conversion rate: **182 missing (4.6%)**
- home fourth-down conversion rate: **159 missing (4.1%)**
- cleaned surface: **26 missing (0.7%)**

The pipeline's imputation strategy allows these rows to remain in the modeling sample rather than discarding a large share of games because of weather or sparse fourth-down information.

## Define Target and Feature Columns

The prediction target is `home_margin`.

To prevent leakage, the feature list excludes game identifiers, final scores, direct outcome variables, spread results, cover/push indicators, and the spread prices used later for betting-profit calculations. The raw `surface` field is also excluded in favor of the cleaned surface category.

`market_home_margin` is intentionally retained because it is known before kickoff and represents the sportsbook's pregame estimate of the final margin.

After these exclusions, the model starts with **194 candidate features**. This broad feature set is evaluated first and then reduced using Random Forest feature importance.

In [13]:
# Target variable
target_col = "home_margin"

# Columns excluded from model features
# These include IDs, final outcomes, direct derivatives of outcomes, odds, and diagnostic columns.
exclude_cols = [
    # identifiers
    "game_id",
    "season",
    "gameday",
    "weekday",
    "home_team",
    "away_team",

    # final game outcomes / leakage
    "away_score",
    "home_score",
    "home_margin",
    "result",
    "total",
    "overtime",

    # spread outcome variables / derivatives
    "spread_result",
    "home_cover",
    "push",
    "home_spread",

    # odds used for backtesting, not score prediction
    "home_spread_odds",
    "away_spread_odds",
    "home_moneyline",
    "away_moneyline",

    # diagnostic columns created during missing-value checks
    "missing_home_feature_count",
    "missing_away_feature_count",

    # use cleaned version instead of raw surface
    "surface"
]

# Keep only columns that actually exist in model_df
exclude_cols = [col for col in exclude_cols if col in model_df.columns]

# Create feature list
feature_cols = [
    col for col in model_df.columns
    if col not in exclude_cols
]

X = model_df[feature_cols].copy()
y = model_df[target_col].copy()

print("Number of model features:", len(feature_cols))
print("Target variable:", target_col)

X.head()

Number of model features: 194
Target variable: home_margin


,week,market_home_margin,total_line,home_rest,away_rest,roof,temp,wind,div_game,stadium,home_qb_id,home_qb_name,home_coach,home_offensive_plays,home_off_epa_per_play,home_off_success_rate,home_yards_per_play,home_pass_rate,home_pass_epa_per_play,home_pass_success_rate,home_yards_per_pass,home_avg_air_yards,home_avg_yac,home_avg_cpoe,home_avg_xpass,home_avg_pass_oe,home_rush_rate,home_rush_epa_per_play,home_rush_success_rate,home_yards_per_rush,home_early_down_pass_rate,home_early_down_pass_success_rate,home_early_down_epa_per_play,home_early_down_pass_epa,home_pressure_allowed_rate,home_sack_rate,home_qb_hit_rate_allowed,home_negative_play_rate,home_turnover_rate,home_explosive_play_rate,home_explosive_pass_rate,home_explosive_rush_rate,home_third_down_attempts,home_third_down_conversions,home_fourth_down_attempts,home_fourth_down_conversions,home_avg_xyac_epa,home_avg_xyac_yards,home_avg_xyac_success,home_third_down_conversion_rate,home_fourth_down_conversion_rate,home_defensive_plays,home_def_epa_allowed_per_play,home_def_success_rate_allowed,home_def_yards_allowed_per_play,home_def_pass_epa_allowed,home_def_pass_success_allowed,home_def_yards_allowed_per_pass,home_def_rush_epa_allowed,home_def_rush_success_allowed,home_def_yards_allowed_per_rush,home_def_early_down_epa_allowed,home_def_early_down_pass_epa_allowed,home_def_early_down_pass_success_allowed,home_pressure_created_rate,home_sack_created_rate,home_qb_hit_created_rate,home_tackle_for_loss_rate,home_explosive_allowed_rate,home_explosive_pass_allowed_rate,home_explosive_rush_allowed_rate,home_turnover_forced_rate,away_qb_id,away_qb_name,away_coach,away_offensive_plays,away_off_epa_per_play,away_off_success_rate,away_yards_per_play,away_pass_rate,away_pass_epa_per_play,away_pass_success_rate,away_yards_per_pass,away_avg_air_yards,away_avg_yac,away_avg_cpoe,away_avg_xpass,away_avg_pass_oe,away_rush_rate,away_rush_epa_per_play,away_rush_success_rate,away_yards_per_rush,away_early_down_pass_rate,away_early_down_pass_success_rate,away_early_down_epa_per_play,away_early_down_pass_epa,away_pressure_allowed_rate,away_sack_rate,away_qb_hit_rate_allowed,away_negative_play_rate,away_turnover_rate,away_explosive_play_rate,away_explosive_pass_rate,away_explosive_rush_rate,away_third_down_attempts,away_third_down_conversions,away_fourth_down_attempts,away_fourth_down_conversions,away_avg_xyac_epa,away_avg_xyac_yards,away_avg_xyac_success,away_third_down_conversion_rate,away_fourth_down_conversion_rate,away_defensive_plays,away_def_epa_allowed_per_play,away_def_success_rate_allowed,away_def_yards_allowed_per_play,away_def_pass_epa_allowed,away_def_pass_success_allowed,away_def_yards_allowed_per_pass,away_def_rush_epa_allowed,away_def_rush_success_allowed,away_def_yards_allowed_per_rush,away_def_early_down_epa_allowed,away_def_early_down_pass_epa_allowed,away_def_early_down_pass_success_allowed,away_pressure_created_rate,away_sack_created_rate,away_qb_hit_created_rate,away_tackle_for_loss_rate,away_explosive_allowed_rate,away_explosive_pass_allowed_rate,away_explosive_rush_allowed_rate,away_turnover_forced_rate,offensive_plays_diff,off_epa_per_play_diff,off_success_rate_diff,yards_per_play_diff,pass_rate_diff,pass_epa_per_play_diff,pass_success_rate_diff,yards_per_pass_diff,avg_air_yards_diff,avg_yac_diff,avg_cpoe_diff,avg_xpass_diff,avg_pass_oe_diff,rush_rate_diff,rush_epa_per_play_diff,rush_success_rate_diff,yards_per_rush_diff,early_down_pass_rate_diff,early_down_pass_success_rate_diff,early_down_epa_per_play_diff,early_down_pass_epa_diff,pressure_allowed_rate_diff,sack_rate_diff,qb_hit_rate_allowed_diff,negative_play_rate_diff,turnover_rate_diff,explosive_play_rate_diff,explosive_pass_rate_diff,explosive_rush_rate_diff,third_down_attempts_diff,third_down_conversions_diff,fourth_down_attempts_diff,fourth_down_conversions_diff,avg_xyac_epa_diff,avg_xyac_yards_diff,avg_xyac_success_diff,third_down_conversion_rate_diff,fourth_down_conversion_rate_diff,defensive_plays_diff,def_epa_al

## Numeric and Categorical Features

The full model feature set contains:

- **185 numeric features**
- **9 categorical features**

The categorical fields include roof/stadium information, quarterback and coach identifiers/names, and the cleaned surface category.

Numeric columns are median-imputed and standardized. Categorical columns are filled with the most frequent category and one-hot encoded. This preprocessing is performed inside the pipeline so the same transformations are applied consistently during training and prediction.

In [14]:
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nCategorical columns:")
categorical_features

Numeric features: 185
Categorical features: 9

Categorical columns:


['roof',
 'stadium',
 'home_qb_id',
 'home_qb_name',
 'home_coach',
 'away_qb_id',
 'away_qb_name',
 'away_coach',
 'surface_clean']

In [15]:
# Make sure no obvious leakage columns accidentally remain
[col for col in feature_cols if "score" in col.lower() or "cover" in col.lower() or "result" in col.lower()]

[]

## Time-Based Train, Validation, and Test Split

A chronological split is used instead of a random split so future seasons do not influence earlier model selection.

The split is:

- **Training:** 2010–2022 → **3,152 games**
- **Validation:** 2023 → **256 games**
- **Final test:** 2024–2025 → **512 games**

The 2023 validation season is used to compare algorithms and feature-set sizes. The 2024–2025 test set is held out until the final model and feature-selection approach have been chosen.

In [16]:
train_df = model_df[model_df["season"] <= 2022].copy()
val_df = model_df[model_df["season"] == 2023].copy()
test_df = model_df[model_df["season"] >= 2024].copy()

X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

print("\nTrain seasons:", train_df["season"].min(), "to", train_df["season"].max())
print("Validation season:", val_df["season"].unique())
print("Test seasons:", sorted(test_df["season"].unique()))

Train shape: (3152, 194)
Validation shape: (256, 194)
Test shape: (512, 194)

Train seasons: 2010 to 2022
Validation season: [2023]
Test seasons: [np.int64(2024), np.int64(2025)]


## Preprocessing Pipeline

Missing values are handled inside the scikit-learn pipeline rather than by deleting additional games.

For numeric variables:

- median imputation fills missing values,
- `add_indicator=True` adds flags showing which values were originally missing,
- standardization places variables on comparable scales.

For categorical variables:

- the most frequent category is used for missing values,
- one-hot encoding converts categories into model-ready numeric columns,
- `handle_unknown="ignore"` allows categories in later seasons that were not present during training.

After Week 1 is removed, the largest remaining missingness is still in weather: **temperature and wind are each missing for 1,248 games (31.8%)**. Fourth-down conversion variables have smaller missingness because some teams have not accumulated enough prior attempts.

In [17]:
# Numeric preprocessing:
# Median imputation handles missing values.
# add_indicator=True creates flags for columns that had missing values.
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler())
    ]
)

# Categorical preprocessing:
# Most frequent imputation handles missing categories.
# One-hot encoding converts categories into numeric model inputs.
try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", onehot)
    ]
)

# Build transformers list
transformers = [
    ("num", numeric_transformer, numeric_features)
]

if len(categorical_features) > 0:
    transformers.append(("cat", categorical_transformer, categorical_features))

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop"
)

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


## Regression Models

Five regression approaches are compared using the same 2010–2022 training period and 2023 validation season:

- **Ridge Regression** — regularized linear regression.
- **Random Forest Regressor** — nonlinear ensemble of decision trees.
- **K-Nearest Neighbors Regressor** — predicts from similar historical observations.
- **Gradient Boosting Regressor** — sequentially improves weak decision trees.
- **HistGradientBoosting Regressor** — histogram-based gradient boosting designed for efficient nonlinear modeling.

Testing several model families helps determine whether the relationship between pregame features and final margin is primarily linear or benefits from nonlinear interactions.

In [18]:
regression_models = {
    "Ridge Regression": Ridge(alpha=1.0),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    ),

    "KNN Regressor": KNeighborsRegressor(
        n_neighbors=25
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "Hist Gradient Boosting": HistGradientBoostingRegressor(
        random_state=42
    )
}

regression_models

{'Ridge Regression': Ridge(),
 'Random Forest': RandomForestRegressor(max_depth=8, min_samples_leaf=10, n_estimators=300,
                       n_jobs=-1, random_state=42),
 'KNN Regressor': KNeighborsRegressor(n_neighbors=25),
 'Gradient Boosting': GradientBoostingRegressor(random_state=42),
 'Hist Gradient Boosting': HistGradientBoostingRegressor(random_state=42)}

## Validation Model Comparison

Each model is trained on 2010–2022 and evaluated on the unseen 2023 season.

Three regression metrics are reported:

- **MAE:** average absolute error in points; lower is better.
- **RMSE:** also measured in points, but penalizes large misses more heavily.
- **R²:** proportion of variation in final margin explained by the model; higher is better.

Among the full-feature machine-learning models, **Random Forest performs best on 2023 with a 10.149 MAE**, followed by Gradient Boosting at **10.443**. The all-feature Ridge model performs poorly at **12.715 MAE and negative R² (-0.247)**, suggesting that the very large feature set adds substantial noise for the linear model.

In [19]:
regression_results = []
trained_models = {}

for model_name, model in regression_models.items():
    print(f"Training {model_name}...")

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    val_preds = pipeline.predict(X_val)

    mae = mean_absolute_error(y_val, val_preds)
    rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    r2 = r2_score(y_val, val_preds)

    regression_results.append({
        "model": model_name,
        "validation_mae": mae,
        "validation_rmse": rmse,
        "validation_r2": r2
    })

    trained_models[model_name] = pipeline

regression_results_df = (
    pd.DataFrame(regression_results)
    .sort_values("validation_mae")
)

regression_results_df

Training Ridge Regression...
Training Random Forest...
Training KNN Regressor...
Training Gradient Boosting...
Training Hist Gradient Boosting...


,model,validation_mae,validation_rmse,validation_r2
1,Random Forest,10.149,13.230,0.140
3,Gradient Boosting,10.443,13.598,0.091
2,KNN Regressor,10.572,13.648,0.084
4,Hist Gradient Boosting,10.637,13.864,0.055
0,Ridge Regression,12.715,15.924,-0.247


In [20]:
fig = px.bar(
    regression_results_df,
    x="model",
    y="validation_mae",
    title="Validation MAE by Regression Model",
    labels={
        "model": "Model",
        "validation_mae": "Validation Mean Absolute Error"
    }
)

fig.show()

## Validation Results Interpretation

Using all 194 features, **Random Forest is the strongest machine-learning model** on the 2023 validation season:

- Random Forest: **10.149 MAE**, 13.230 RMSE, 0.140 R²
- Gradient Boosting: **10.443 MAE**
- KNN: **10.572 MAE**
- Hist Gradient Boosting: **10.637 MAE**
- Ridge Regression: **12.715 MAE**

The wide gap between the full-feature Ridge model and the tree-based models motivates the feature-selection experiment below. The result suggests that Ridge may work better after noisy or redundant variables are removed.

Prediction accuracy and betting profitability are evaluated separately. A model can have a strong MAE without generating useful spread disagreements, and a profitable betting result can occur even when the sportsbook remains the better overall margin predictor.

## Sportsbook Baseline Comparison

The sportsbook baseline simply uses `market_home_margin` as the predicted final home margin.

On the 2023 validation season, the sportsbook baseline produces:

- **MAE: 9.818**
- **RMSE: 12.957**
- **R²: 0.175**

This is better than every full-feature machine-learning model. Random Forest is the closest ML model at **10.149 MAE**, about **0.33 points worse per game**.

This establishes a demanding benchmark: the betting market is already a strong predictor of final margin, so the ML model does not need to replace the market on every game. Its potential value may instead come from identifying selective games where its estimate differs meaningfully from the sportsbook.

In [21]:
# Baseline prediction: sportsbook expected home margin
market_val_preds = val_df["market_home_margin"]

market_mae = mean_absolute_error(y_val, market_val_preds)
market_rmse = np.sqrt(mean_squared_error(y_val, market_val_preds))
market_r2 = r2_score(y_val, market_val_preds)

market_baseline_row = pd.DataFrame([{
    "model": "Sportsbook Baseline",
    "validation_mae": market_mae,
    "validation_rmse": market_rmse,
    "validation_r2": market_r2
}])

comparison_results_df = pd.concat(
    [regression_results_df, market_baseline_row],
    ignore_index=True
).sort_values("validation_mae")

comparison_results_df

,model,validation_mae,validation_rmse,validation_r2
5,Sportsbook Baseline,9.818,12.957,0.175
0,Random Forest,10.149,13.230,0.140
1,Gradient Boosting,10.443,13.598,0.091
2,KNN Regressor,10.572,13.648,0.084
3,Hist Gradient Boosting,10.637,13.864,0.055
4,Ridge Regression,12.715,15.924,-0.247


## Feature Selection Experiment

The initial model comparison uses all **194 features**. This section tests whether a smaller feature set can reduce noise and improve validation accuracy.

A Random Forest is trained **only on the 2010–2022 training data** using numeric predictors. Its feature importances are then used to rank variables before looking at the 2023 validation results.

`market_home_margin` is by far the dominant feature with an importance of approximately **0.373**. The remaining individual importances are much smaller, with several team-efficiency variables clustered near 0.005–0.006.

The experiment compares the top 20, 40, and 60 ranked numeric features. Because selection is based only on the training period, the validation and test seasons are not used to choose the variables.

In [22]:
# Numeric-only Random Forest for feature screening
# This avoids complications from one-hot encoded categorical variables.

feature_screening_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

feature_screening_pipeline.fit(X_train[numeric_features], y_train)

feature_importance_df = pd.DataFrame({
    "feature": numeric_features,
    "importance": feature_screening_pipeline.named_steps["model"].feature_importances_
}).sort_values("importance", ascending=False)

feature_importance_df.head(40)

,feature,importance
1,market_home_margin,0.373
153,explosive_pass_rate_diff,0.006
169,def_pass_success_allowed_diff,0.006
8,home_offensive_plays,0.006
141,rush_success_rate_diff,0.006
113,away_def_rush_success_allowed,0.006
77,away_avg_cpoe,0.006
46,home_defensive_plays,0.006
136,avg_cpoe_diff,0.006
166,def_success_rate_allowed_diff,0.006


In [23]:
# Numeric-only Random Forest for feature screening
# This avoids complications from one-hot encoded categorical variables.

feature_screening_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

feature_screening_pipeline.fit(X_train[numeric_features], y_train)

feature_importance_df = pd.DataFrame({
    "feature": numeric_features,
    "importance": feature_screening_pipeline.named_steps["model"].feature_importances_
}).sort_values("importance", ascending=False)

feature_importance_df.head(40)

,feature,importance
1,market_home_margin,0.373
153,explosive_pass_rate_diff,0.006
169,def_pass_success_allowed_diff,0.006
8,home_offensive_plays,0.006
141,rush_success_rate_diff,0.006
113,away_def_rush_success_allowed,0.006
77,away_avg_cpoe,0.006
46,home_defensive_plays,0.006
136,avg_cpoe_diff,0.006
166,def_success_rate_allowed_diff,0.006


In [24]:
top_importance = feature_importance_df.head(40)

fig = px.bar(
    top_importance,
    x="importance",
    y="feature",
    orientation="h",
    title="Top 40 Numeric Feature Importances from Random Forest",
    labels={
        "importance": "Feature Importance",
        "feature": "Feature"
    }
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

### Feature Importance Takeaway

The Random Forest screening model assigns approximately **37.3% of its total importance to `market_home_margin`**, making the sportsbook expectation the dominant individual predictor by a wide margin.

The next-ranked variables—such as `explosive_pass_rate_diff`, `def_pass_success_allowed_diff`, `home_offensive_plays`, and `rush_success_rate_diff`—each contribute much smaller individual importances around **0.6%**.

This does not mean the remaining football features are unimportant collectively. Rather, it shows that the market already summarizes a large amount of pregame information, while the advanced team variables provide smaller incremental signals that may help refine that baseline.

## Reduced Feature Set Model Comparison

The Random Forest ranking is used to test three reduced feature sets:

- Top 20 RF features
- Top 40 RF features
- Top 60 RF features

`market_home_margin` is forced into each set if necessary.

Feature reduction materially improves Ridge Regression. With all 194 features, Ridge had a **12.715 validation MAE**; with the top 20 features, its MAE falls to **10.030**, making it the best machine-learning configuration tested.

The top-20 Ridge result is also much closer to the sportsbook baseline of **9.818 MAE**. This supports the idea that a smaller set of stronger signals is more useful for the regularized linear model than the full high-dimensional feature set.

In [25]:
def evaluate_models_with_feature_set(selected_features, feature_set_name):
    """
    Train and evaluate regression models using a selected feature set.
    """
    X_train_fs = train_df[selected_features].copy()
    X_val_fs = val_df[selected_features].copy()

    numeric_fs = X_train_fs.select_dtypes(include=["number"]).columns.tolist()
    categorical_fs = X_train_fs.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    numeric_transformer_fs = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler())
        ]
    )

    try:
        onehot_fs = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        onehot_fs = OneHotEncoder(handle_unknown="ignore", sparse=False)

    categorical_transformer_fs = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", onehot_fs)
        ]
    )

    transformers_fs = [
        ("num", numeric_transformer_fs, numeric_fs)
    ]

    if len(categorical_fs) > 0:
        transformers_fs.append(("cat", categorical_transformer_fs, categorical_fs))

    preprocessor_fs = ColumnTransformer(
        transformers=transformers_fs,
        remainder="drop"
    )

    results = []

    for model_name, model in regression_models.items():
        pipeline = Pipeline(
            steps=[
                ("preprocessor", preprocessor_fs),
                ("model", model)
            ]
        )

        pipeline.fit(X_train_fs, y_train)
        val_preds = pipeline.predict(X_val_fs)

        results.append({
            "feature_set": feature_set_name,
            "num_features": len(selected_features),
            "model": model_name,
            "validation_mae": mean_absolute_error(y_val, val_preds),
            "validation_rmse": np.sqrt(mean_squared_error(y_val, val_preds)),
            "validation_r2": r2_score(y_val, val_preds)
        })

    return pd.DataFrame(results)

In [26]:
feature_set_results = []

for n in [20, 40, 60]:
    selected_features = feature_importance_df.head(n)["feature"].tolist()

    # Force market_home_margin into the feature set if it is not already there
    if "market_home_margin" in feature_cols and "market_home_margin" not in selected_features:
        selected_features = ["market_home_margin"] + selected_features

    fs_results = evaluate_models_with_feature_set(
        selected_features=selected_features,
        feature_set_name=f"Top {n} RF Features"
    )

    feature_set_results.append(fs_results)

feature_set_results_df = pd.concat(feature_set_results, ignore_index=True)

feature_set_results_df.sort_values("validation_mae").head(20)

,feature_set,num_features,model,validation_mae,validation_rmse,validation_r2
0,Top 20 RF Features,20,Ridge Regression,10.030,13.060,0.162
6,Top 40 RF Features,40,Random Forest,10.160,13.278,0.133
11,Top 60 RF Features,60,Random Forest,10.180,13.257,0.136
8,Top 40 RF Features,40,Gradient Boosting,10.247,13.472,0.108
5,Top 40 RF Features,40,Ridge Regression,10.247,13.185,0.145
10,Top 60 RF Features,60,Ridge Regression,10.253,13.149,0.150
1,Top 20 RF Features,20,Random Forest,10.352,13.412,0.116
3,Top 20 RF Features,20,Gradient Boosting,10.362,13.366,0.122
13,Top 60 RF Features,60,Gradient Boosting,10.362,13.427,0.114
7,Top 40 RF Features,40,KNN Regressor,10.531,13.608,0.090


In [27]:
all_features_labeled = regression_results_df.copy()
all_features_labeled["feature_set"] = "All Features"
all_features_labeled["num_features"] = len(feature_cols)

all_feature_comparison_df = pd.concat(
    [
        all_features_labeled[
            ["feature_set", "num_features", "model", "validation_mae", "validation_rmse", "validation_r2"]
        ],
        feature_set_results_df
    ],
    ignore_index=True
)

all_feature_comparison_df.sort_values("validation_mae").head(25)

,feature_set,num_features,model,validation_mae,validation_rmse,validation_r2
5,Top 20 RF Features,20,Ridge Regression,10.030,13.060,0.162
0,All Features,194,Random Forest,10.149,13.230,0.140
11,Top 40 RF Features,40,Random Forest,10.160,13.278,0.133
16,Top 60 RF Features,60,Random Forest,10.180,13.257,0.136
13,Top 40 RF Features,40,Gradient Boosting,10.247,13.472,0.108
10,Top 40 RF Features,40,Ridge Regression,10.247,13.185,0.145
15,Top 60 RF Features,60,Ridge Regression,10.253,13.149,0.150
6,Top 20 RF Features,20,Random Forest,10.352,13.412,0.116
8,Top 20 RF Features,20,Gradient Boosting,10.362,13.366,0.122
18,Top 60 RF Features,60,Gradient Boosting,10.362,13.427,0.114


In [28]:
fig = px.bar(
    all_feature_comparison_df.sort_values("validation_mae"),
    x="model",
    y="validation_mae",
    color="feature_set",
    barmode="group",
    title="Validation MAE by Model and Feature Set",
    labels={
        "model": "Model",
        "validation_mae": "Validation MAE",
        "feature_set": "Feature Set"
    }
)

fig.show()

## Final Model Selection and Implementation

The feature-set comparison identifies **Ridge Regression with the top 20 Random Forest-ranked features** as the best validation configuration, with:

- **MAE: 10.030**
- **RMSE: 13.060**
- **R²: 0.162**

The sportsbook baseline remains slightly better on 2023 at **9.818 MAE**.

**Important implementation detail:** the following code cell selects `feature_importance_df.head(5)`, not the top 20. Therefore, the downstream final holdout model and its betting backtest actually use **five selected numeric features**, even though several printed labels still say “Top 20 RF Features + Ridge Regression.”

Because the instruction for this notebook is to preserve the code exactly, the markdown documents the model that was actually executed rather than changing that implementation.

In [29]:
# Select the top 20 features from Random Forest feature importance
selected_feature_cols = feature_importance_df.head(5)["feature"].tolist()

# Make sure market_home_margin is included
if "market_home_margin" in feature_cols and "market_home_margin" not in selected_feature_cols:
    selected_feature_cols = ["market_home_margin"] + selected_feature_cols

print("Number of selected features:", len(selected_feature_cols))
selected_feature_cols

Number of selected features: 5


['market_home_margin',
 'explosive_pass_rate_diff',
 'def_pass_success_allowed_diff',
 'home_offensive_plays',
 'rush_success_rate_diff']

## Final Training and Test Split

After model selection, the final Ridge pipeline is trained on both the original training and validation periods.

The downstream implementation uses the **five features selected in the previous code cell**:

1. `market_home_margin`
2. `explosive_pass_rate_diff`
3. `def_pass_success_allowed_diff`
4. `home_offensive_plays`
5. `rush_success_rate_diff`

Final split:

- **Training:** 2010–2023 → **3,408 games**
- **Testing:** 2024–2025 → **512 games**

The test seasons remain unseen during this final fitting step.

In [30]:
# Final training data uses both train and validation seasons
final_train_df = model_df[model_df["season"] <= 2023].copy()
final_test_df = model_df[model_df["season"] >= 2024].copy()

X_final_train = final_train_df[selected_feature_cols].copy()
y_final_train = final_train_df[target_col].copy()

X_final_test = final_test_df[selected_feature_cols].copy()
y_final_test = final_test_df[target_col].copy()

print("Final train shape:", X_final_train.shape)
print("Final test shape:", X_final_test.shape)

print("Final train seasons:", final_train_df["season"].min(), "to", final_train_df["season"].max())
print("Final test seasons:", sorted(final_test_df["season"].unique()))

Final train shape: (3408, 5)
Final test shape: (512, 5)
Final train seasons: 2010 to 2023
Final test seasons: [np.int64(2024), np.int64(2025)]


## Final Model Pipeline

The final holdout model is Ridge Regression with the **five numeric features actually selected by the code**. No categorical variables are included in this reduced set.

The pipeline:

1. median-imputes missing numeric values,
2. adds missing-value indicators where needed,
3. standardizes the selected variables,
4. fits Ridge Regression with `alpha=1.0`.

The code output labels this model as “Top 20 RF Features + Ridge Regression,” but the executed final pipeline contains five selected features. This distinction is important when interpreting the holdout and betting results below.

In [31]:
# Identify numeric and categorical columns in the selected feature set
selected_numeric_features = X_final_train.select_dtypes(include=["number"]).columns.tolist()
selected_categorical_features = X_final_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Selected numeric features:", len(selected_numeric_features))
print("Selected categorical features:", len(selected_categorical_features))
print("Categorical features:", selected_categorical_features)

Selected numeric features: 5
Selected categorical features: 0
Categorical features: []


In [32]:
# Numeric preprocessing
selected_numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler())
    ]
)

# Categorical preprocessing
try:
    selected_onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    selected_onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

selected_categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", selected_onehot)
    ]
)

selected_transformers = [
    ("num", selected_numeric_transformer, selected_numeric_features)
]

if len(selected_categorical_features) > 0:
    selected_transformers.append(
        ("cat", selected_categorical_transformer, selected_categorical_features)
    )

selected_preprocessor = ColumnTransformer(
    transformers=selected_transformers,
    remainder="drop"
)

final_model = Pipeline(
    steps=[
        ("preprocessor", selected_preprocessor),
        ("model", Ridge(alpha=1.0))
    ]
)

final_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['market_home_margin',
                                                   'explosive_pass_rate_diff',
                                                   'def_pass_success_allowed_diff',
                                                   'home_offensive_plays',
                                                   'rush_success_rate_diff'])])),
                ('model', Ridge())])

In [33]:
# Train final model
final_model.fit(X_final_train, y_final_train)

# Predict test set
final_test_preds = final_model.predict(X_final_test)

# Evaluate test performance
final_test_mae = mean_absolute_error(y_final_test, final_test_preds)
final_test_rmse = np.sqrt(mean_squared_error(y_final_test, final_test_preds))
final_test_r2 = r2_score(y_final_test, final_test_preds)

print("Final Model Test Performance")
print("----------------------------")
print("Model: Top 20 RF Features + Ridge Regression")
print(f"MAE: {final_test_mae:.3f}")
print(f"RMSE: {final_test_rmse:.3f}")
print(f"R2: {final_test_r2:.3f}")

Final Model Test Performance
----------------------------
Model: Top 20 RF Features + Ridge Regression
MAE: 9.774
RMSE: 12.465
R2: 0.256


## Test Set Sportsbook Baseline

The 2024–2025 holdout compares the final executed Ridge model with the sportsbook baseline.

Results:

| Method | MAE | RMSE | R² |
|---|---:|---:|---:|
| Final Ridge implementation | **9.774** | **12.465** | **0.256** |
| Sportsbook baseline | 9.804 | 12.531 | 0.248 |

The final model improves MAE by only about **0.03 points per game**, but it also has a slightly lower RMSE and slightly higher R². The difference is small, so the main question becomes whether the model's selective disagreements with the market are useful for betting rather than whether it decisively outpredicts the sportsbook overall.

In [34]:
# Sportsbook baseline on final test set
market_test_preds = final_test_df["market_home_margin"]

market_test_mae = mean_absolute_error(y_final_test, market_test_preds)
market_test_rmse = np.sqrt(mean_squared_error(y_final_test, market_test_preds))
market_test_r2 = r2_score(y_final_test, market_test_preds)

test_comparison_df = pd.DataFrame([
    {
        "model": "Sportsbook Baseline",
        "test_mae": market_test_mae,
        "test_rmse": market_test_rmse,
        "test_r2": market_test_r2
    },
    {
        "model": "Top 20 RF Features + Ridge Regression",
        "test_mae": final_test_mae,
        "test_rmse": final_test_rmse,
        "test_r2": final_test_r2
    }
]).sort_values("test_mae")

test_comparison_df

,model,test_mae,test_rmse,test_r2
1,Top 20 RF Features + Ridge Regression,9.774,12.465,0.256
0,Sportsbook Baseline,9.804,12.531,0.248


### Final Holdout Prediction Takeaway

On the unseen 2024–2025 test set, the executed five-feature Ridge model is **slightly more accurate than the sportsbook baseline**:

- MAE improves from **9.804 to 9.774**
- RMSE improves from **12.531 to 12.465**
- R² increases from **0.248 to 0.256**

The improvement is real within this test sample but very small—only about **0.03 MAE points per game**. The betting backtest therefore becomes an important second evaluation: it tests whether the model's relatively rare disagreements with the market are more informative than the small aggregate accuracy difference suggests.

## Final Prediction Dataset

For each 2024–2025 test game, the notebook stores:

- actual `home_margin`,
- sportsbook `market_home_margin`,
- model `predicted_home_margin`,
- `model_edge = predicted_home_margin - market_home_margin`.

This game-level table is the bridge between regression and betting. The model predicts margin first; the threshold rule then determines whether the disagreement is large enough to create a simulated bet.

Because the final Ridge implementation is heavily anchored by `market_home_margin`, most predictions remain close to the sportsbook line. As a result, larger betting thresholds produce relatively few bets in the 2024–2025 holdout.

In [35]:
final_test_results = final_test_df.copy()

final_test_results["predicted_home_margin"] = final_test_preds

final_test_results["model_edge"] = (
    final_test_results["predicted_home_margin"] -
    final_test_results["market_home_margin"]
)

final_test_results[
    [
        "season",
        "week",
        "home_team",
        "away_team",
        "home_margin",
        "market_home_margin",
        "predicted_home_margin",
        "model_edge",
        "home_spread_odds",
        "away_spread_odds"
    ]
].head(20)

,season,week,home_team,away_team,home_margin,market_home_margin,predicted_home_margin,model_edge,home_spread_odds,away_spread_odds
3647,2024,2,MIA,BUF,-21.000,2.500,1.355,-1.145,-110.000,-110.000
3648,2024,2,BAL,LV,-3.000,9.000,14.994,5.994,-108.000,-112.000
3649,2024,2,CAR,LAC,-23.000,-5.000,-6.411,-1.411,-110.000,-110.000
3650,2024,2,DAL,NO,-25.000,6.500,5.271,-1.229,-108.000,-112.000
3651,2024,2,DET,TB,-4.000,7.500,9.800,2.300,-112.000,-108.000
3652,2024,2,GB,IND,6.000,-3.000,-3.128,-0.128,-118.000,-102.000
3653,2024,2,JAX,CLE,-5.000,3.000,1.458,-1.542,-120.000,100.000
3654,2024,2,MIN,SF,6.000,-4.500,-7.431,-2.931,-108.000,-112.000
3655,2024,2,NE,SEA,-3.000,-3.500,-2.349,1.151,-118.000,-102.000
3656,2024,2,TEN,NYJ,-7.000,-3.500,-4.033,-0.533,-108.000,-112.000


## Betting Backtest Logic

The betting edge is:

`model_edge = predicted_home_margin - market_home_margin`

Decision rule:

- `model_edge >= threshold` → bet home ATS
- `model_edge <= -threshold` → bet away ATS
- otherwise → no bet

Each wager risks **$100**. Winning profit is calculated from the actual American spread price for the selected side; losses are `-$100`; pushes return `$0` profit. Games with a qualifying edge but missing spread odds are labeled `no_odds` and are not included in profit.

The bet is graded against the market margin, not the predicted margin: a home bet wins when actual `home_margin` exceeds `market_home_margin`, while an away bet wins when it finishes below that market expectation.

In [36]:
def american_odds_profit(odds, stake=100):
    """
    Calculate profit on a winning bet using American odds.

    For example:
    -110 odds means a $100 bet wins $90.91 in profit.
    +120 odds means a $100 bet wins $120 in profit.
    """
    if pd.isna(odds):
        return np.nan

    if odds > 0:
        return stake * (odds / 100)
    else:
        return stake * (100 / abs(odds))

In [37]:
def backtest_spread_bets(results_df, threshold, stake=100):
    """
    Backtest spread bets using model edge.

    Positive model_edge means the model likes the home team.
    Negative model_edge means the model likes the away team.
    """
    bt = results_df.copy()

    # Determine bet side based on model edge
    bt["bet_side"] = np.where(
        bt["model_edge"] >= threshold,
        "home",
        np.where(bt["model_edge"] <= -threshold, "away", "no_bet")
    )

    # Assign the correct spread odds based on selected side
    bt["bet_odds"] = np.where(
        bt["bet_side"] == "home",
        bt["home_spread_odds"],
        np.where(bt["bet_side"] == "away", bt["away_spread_odds"], np.nan)
    )

    # If odds are missing, we cannot calculate profit correctly
    bt.loc[
        (bt["bet_side"].isin(["home", "away"])) &
        (bt["bet_odds"].isna()),
        "bet_side"
    ] = "no_odds"

    # Default result
    bt["bet_result"] = "no_bet"

    # Home bet wins if actual home margin is greater than market expected home margin
    bt.loc[
        (bt["bet_side"] == "home") &
        (bt["home_margin"] > bt["market_home_margin"]),
        "bet_result"
    ] = "win"

    bt.loc[
        (bt["bet_side"] == "home") &
        (bt["home_margin"] < bt["market_home_margin"]),
        "bet_result"
    ] = "loss"

    # Away bet wins if actual home margin is less than market expected home margin
    bt.loc[
        (bt["bet_side"] == "away") &
        (bt["home_margin"] < bt["market_home_margin"]),
        "bet_result"
    ] = "win"

    bt.loc[
        (bt["bet_side"] == "away") &
        (bt["home_margin"] > bt["market_home_margin"]),
        "bet_result"
    ] = "loss"

    # Push if actual margin equals market margin
    bt.loc[
        (bt["bet_side"].isin(["home", "away"])) &
        (bt["home_margin"] == bt["market_home_margin"]),
        "bet_result"
    ] = "push"

    bt.loc[bt["bet_side"] == "no_odds", "bet_result"] = "no_odds"

    # Calculate profit
    bt["profit"] = 0.0

    bt.loc[bt["bet_result"] == "loss", "profit"] = -stake
    bt.loc[bt["bet_result"] == "push", "profit"] = 0

    win_mask = bt["bet_result"] == "win"

    bt.loc[win_mask, "profit"] = bt.loc[win_mask, "bet_odds"].apply(
        lambda odds: american_odds_profit(odds, stake)
    )

    return bt

In [38]:
selected_threshold = 2.5

backtest_25 = backtest_spread_bets(
    final_test_results,
    threshold=selected_threshold,
    stake=100
)

backtest_25[
    [
        "season",
        "week",
        "home_team",
        "away_team",
        "home_margin",
        "market_home_margin",
        "predicted_home_margin",
        "model_edge",
        "bet_side",
        "bet_odds",
        "bet_result",
        "profit"
    ]
].head(30)

,season,week,home_team,away_team,home_margin,market_home_margin,predicted_home_margin,model_edge,bet_side,bet_odds,bet_result,profit
3647,2024,2,MIA,BUF,-21.000,2.500,1.355,-1.145,no_bet,NaN,no_bet,0.000
3648,2024,2,BAL,LV,-3.000,9.000,14.994,5.994,home,-108.000,loss,-100.000
3649,2024,2,CAR,LAC,-23.000,-5.000,-6.411,-1.411,no_bet,NaN,no_bet,0.000
3650,2024,2,DAL,NO,-25.000,6.500,5.271,-1.229,no_bet,NaN,no_bet,0.000
3651,2024,2,DET,TB,-4.000,7.500,9.800,2.300,no_bet,NaN,no_bet,0.000
3652,2024,2,GB,IND,6.000,-3.000,-3.128,-0.128,no_bet,NaN,no_bet,0.000
3653,2024,2,JAX,CLE,-5.000,3.000,1.458,-1.542,no_bet,NaN,no_bet,0.000
3654,2024,2,MIN,SF,6.000,-4.500,-7.431,-2.931,away,-112.000,loss,-100.000
3655,2024,2,NE,SEA,-3.000,-3.500,-2.349,1.151,no_bet,NaN,no_bet,0.000
3656,2024,2,TEN,NYJ,-7.000,-3.500,-4.033,-0.533,no_bet,NaN,no_bet,0.000


In [39]:
bets_only = backtest_25[backtest_25["bet_side"].isin(["home", "away"])].copy()

total_bets = len(bets_only)
wins = (bets_only["bet_result"] == "win").sum()
losses = (bets_only["bet_result"] == "loss").sum()
pushes = (bets_only["bet_result"] == "push").sum()
total_profit = bets_only["profit"].sum()
total_risked = total_bets * 100
roi = total_profit / total_risked if total_risked > 0 else np.nan
win_rate = wins / (wins + losses) if (wins + losses) > 0 else np.nan

print(f"Threshold: {selected_threshold}")
print(f"Total bets: {total_bets}")
print(f"Wins: {wins}")
print(f"Losses: {losses}")
print(f"Pushes: {pushes}")
print(f"Win rate excluding pushes: {win_rate:.3f}")
print(f"Total profit: ${total_profit:.2f}")
print(f"ROI: {roi:.3f}")

Threshold: 2.5
Total bets: 7
Wins: 5
Losses: 2
Pushes: 0
Win rate excluding pushes: 0.714
Total profit: $261.58
ROI: 0.374


## Threshold Backtesting

The 2024–2025 holdout is evaluated at edge thresholds from **0.5 through 10.5 points**.

A low threshold places more bets because even a small model-market disagreement qualifies. A high threshold requires a stronger disagreement but sharply reduces the sample.

This tradeoff is especially important for this model because its predictions tend to stay close to the sportsbook line. At a 0.5-point threshold there are **269 bets**, while only **7 bets** qualify at 2.5 points and **3 bets** qualify at 3.5 points.

In [40]:
thresholds = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5]

threshold_results = []

for threshold in thresholds:
    bt = backtest_spread_bets(
        final_test_results,
        threshold=threshold,
        stake=100
    )

    bets = bt[bt["bet_side"].isin(["home", "away"])].copy()

    total_bets = len(bets)
    wins = (bets["bet_result"] == "win").sum()
    losses = (bets["bet_result"] == "loss").sum()
    pushes = (bets["bet_result"] == "push").sum()
    no_odds = (bt["bet_result"] == "no_odds").sum()

    total_profit = bets["profit"].sum()
    total_risked = total_bets * 100
    roi = total_profit / total_risked if total_risked > 0 else np.nan
    win_rate = wins / (wins + losses) if (wins + losses) > 0 else np.nan

    threshold_results.append({
        "model": "Top 20 RF Features + Ridge Regression",
        "threshold": threshold,
        "total_bets": total_bets,
        "wins": wins,
        "losses": losses,
        "pushes": pushes,
        "no_odds_games": no_odds,
        "win_rate_excluding_pushes": win_rate,
        "total_profit": total_profit,
        "total_risked": total_risked,
        "roi": roi
    })

threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df

,model,threshold,total_bets,wins,losses,pushes,no_odds_games,win_rate_excluding_pushes,total_profit,total_risked,roi
0,Top 20 RF Features + Ridge Regression,0.500,269,143,123,3,0,0.538,724.532,26900,0.027
1,Top 20 RF Features + Ridge Regression,1.500,44,20,24,0,0,0.455,-559.204,4400,-0.127
2,Top 20 RF Features + Ridge Regression,2.500,7,5,2,0,0,0.714,261.580,700,0.374
3,Top 20 RF Features + Ridge Regression,3.500,3,2,1,0,0,0.667,90.476,300,0.302
4,Top 20 RF Features + Ridge Regression,4.500,1,0,1,0,0,0.000,-100.000,100,-1.000
5,Top 20 RF Features + Ridge Regression,5.500,1,0,1,0,0,0.000,-100.000,100,-1.000
6,Top 20 RF Features + Ridge Regression,6.500,0,0,0,0,0,NaN,0.000,0,NaN
7,Top 20 RF Features + Ridge Regression,7.500,0,0,0,0,0,NaN,0.000,0,NaN
8,Top 20 RF Features + Ridge Regression,8.500,0,0,0,0,0,NaN,0.000,0,NaN
9,Top 20 RF Features + Ridge Regression,9.500,0,0,0,0,0,NaN,0.000,0,NaN


In [41]:
fig = px.line(
    threshold_results_df,
    x="threshold",
    y="roi",
    markers=True,
    title="ROI by Betting Threshold",
    labels={
        "threshold": "Model Edge Threshold",
        "roi": "ROI"
    }
)

fig.show()

In [42]:
fig = px.line(
    threshold_results_df,
    x="threshold",
    y="total_profit",
    markers=True,
    title="Total Profit by Betting Threshold",
    labels={
        "threshold": "Model Edge Threshold",
        "total_profit": "Total Profit ($)"
    }
)

fig.show()

In [43]:
fig = px.bar(
    threshold_results_df,
    x="threshold",
    y="total_bets",
    title="Number of Bets by Threshold",
    labels={
        "threshold": "Model Edge Threshold",
        "total_bets": "Number of Bets"
    }
)

fig.show()

## Threshold Results Interpretation

The 2024–2025 holdout does **not** show a simple relationship where every larger edge performs better:

- **0.5 edge:** 269 bets, 53.8% win rate, **+$724.53**, **2.7% ROI**
- **1.5 edge:** 44 bets, 45.5% win rate, **-$559.20**, **-12.7% ROI**
- **2.5 edge:** 7 bets, 71.4% win rate, **+$261.58**, **37.4% ROI**
- **3.5 edge:** 3 bets, 66.7% win rate, **+$90.48**, **30.2% ROI**
- 4.5 and 5.5 each contain only one losing bet; no games reach thresholds of 6.5 or higher.

The 2.5- and 3.5-point results look strong, but their sample sizes are too small to support a strong conclusion from the holdout alone. This is why the later walk-forward analysis is more informative: it applies the same threshold logic across many historical seasons.

## Walk-Forward Season Backtest

The holdout analysis evaluates only 2024–2025. To test historical stability, this section runs a season-by-season walk-forward backtest from **2011 through 2025**.

For each test season:

1. only earlier seasons are used for training,
2. Random Forest feature importance is recalculated using that historical training sample,
3. the **top 20 features** are selected for that season,
4. Ridge Regression is fitted,
5. the test season is predicted,
6. betting thresholds from 0.5 to 4.5 points are evaluated.

For example, the 2015 model trains on 2010–2014, while the 2025 model trains on 2010–2024.

**Important distinction:** unlike the five-feature final holdout implementation above, the walk-forward code explicitly selects **20 features for each historical season**. Therefore, the walk-forward/scenario results describe a top-20 Ridge workflow, while the 2024–2025 final-model results above describe the five-feature implementation actually executed in that section.

In [57]:
def summarize_backtest(bt, model_name, threshold, test_season):
    """
    Summarize betting results for one model, threshold, and test season.
    """
    bets = bt[bt["bet_side"].isin(["home", "away"])].copy()

    total_bets = len(bets)
    wins = (bets["bet_result"] == "win").sum()
    losses = (bets["bet_result"] == "loss").sum()
    pushes = (bets["bet_result"] == "push").sum()
    total_profit = bets["profit"].sum()
    total_risked = total_bets * 100

    win_rate = wins / (wins + losses) if (wins + losses) > 0 else np.nan
    roi = total_profit / total_risked if total_risked > 0 else np.nan

    return {
        "model": model_name,
        "test_season": test_season,
        "threshold": threshold,
        "total_bets": total_bets,
        "wins": wins,
        "losses": losses,
        "pushes": pushes,
        "win_rate_excluding_pushes": win_rate,
        "total_profit": total_profit,
        "total_risked": total_risked,
        "roi": roi
    }

In [58]:
walk_forward_thresholds = [0.5, 1.5, 2.5, 3.5, 4.5]

walk_forward_results = []
walk_forward_predictions = []

# Cannot test 2010 because there is no prior season to train on
test_seasons = sorted(model_df["season"].unique())
test_seasons = [season for season in test_seasons if season > model_df["season"].min()]

for test_season in test_seasons:
    print(f"Running walk-forward backtest for {test_season}...")

    # Train only on seasons before the test season
    train_season_df = model_df[model_df["season"] < test_season].copy()
    test_season_df = model_df[model_df["season"] == test_season].copy()

    # Skip if there is not enough data
    if train_season_df.shape[0] == 0 or test_season_df.shape[0] == 0:
        continue

    # Use the same feature pool as before
    X_train_season_all = train_season_df[feature_cols].copy()
    y_train_season = train_season_df[target_col].copy()

    X_test_season_all = test_season_df[feature_cols].copy()
    y_test_season = test_season_df[target_col].copy()

    # Numeric features for Random Forest feature selection
    season_numeric_features = X_train_season_all.select_dtypes(include=["number"]).columns.tolist()

    # Feature screening model trained only on prior seasons
    season_feature_screening_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(
                n_estimators=300,
                max_depth=8,
                min_samples_leaf=10,
                random_state=42,
                n_jobs=-1
            ))
        ]
    )

    season_feature_screening_pipeline.fit(
        X_train_season_all[season_numeric_features],
        y_train_season
    )

    season_feature_importance_df = pd.DataFrame({
        "feature": season_numeric_features,
        "importance": season_feature_screening_pipeline.named_steps["model"].feature_importances_
    }).sort_values("importance", ascending=False)

    # Select top 20 features
    season_selected_features = season_feature_importance_df.head(20)["feature"].tolist()

    # Force market_home_margin into the feature set
    if "market_home_margin" in feature_cols and "market_home_margin" not in season_selected_features:
        season_selected_features = ["market_home_margin"] + season_selected_features

    X_train_season = train_season_df[season_selected_features].copy()
    X_test_season = test_season_df[season_selected_features].copy()

    # Identify selected feature types
    season_numeric_selected = X_train_season.select_dtypes(include=["number"]).columns.tolist()
    season_categorical_selected = X_train_season.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    # Numeric preprocessing
    season_numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler())
        ]
    )

    # Categorical preprocessing
    try:
        season_onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        season_onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

    season_categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", season_onehot)
        ]
    )

    season_transformers = [
        ("num", season_numeric_transformer, season_numeric_selected)
    ]

    if len(season_categorical_selected) > 0:
        season_transformers.append(
            ("cat", season_categorical_transformer, season_categorical_selected)
        )

    season_preprocessor = ColumnTransformer(
        transformers=season_transformers,
        remainder="drop"
    )

    # Final model for this season
    season_model = Pipeline(
        steps=[
            ("preprocessor", season_preprocessor),
            ("model", Ridge(alpha=1.0))
        ]
    )

    # Train and predict
    season_model.fit(X_train_season, y_train_season)
    season_preds = season_model.predict(X_test_season)

    # Prediction performance
    season_mae = mean_absolute_error(y_test_season, season_preds)
    season_rmse = np.sqrt(mean_squared_error(y_test_season, season_preds))
    season_r2 = r2_score(y_test_season, season_preds)

    # Create results dataframe for betting backtest
    season_results = test_season_df.copy()
    season_results["predicted_home_margin"] = season_preds
    season_results["model_edge"] = (
        season_results["predicted_home_margin"] -
        season_results["market_home_margin"]
    )

    season_results["walk_forward_model"] = "Top 20 RF Features + Ridge Regression"
    season_results["test_season"] = test_season
    season_results["season_mae"] = season_mae
    season_results["season_rmse"] = season_rmse
    season_results["season_r2"] = season_r2

    walk_forward_predictions.append(season_results)

    # Backtest each threshold
    for threshold in walk_forward_thresholds:
        bt = backtest_spread_bets(
            season_results,
            threshold=threshold,
            stake=100
        )

        summary = summarize_backtest(
            bt=bt,
            model_name="Top 20 RF Features + Ridge Regression",
            threshold=threshold,
            test_season=test_season
        )

        summary["season_mae"] = season_mae
        summary["season_rmse"] = season_rmse
        summary["season_r2"] = season_r2
        summary["num_selected_features"] = len(season_selected_features)

        walk_forward_results.append(summary)

walk_forward_results_df = pd.DataFrame(walk_forward_results)
walk_forward_predictions_df = pd.concat(walk_forward_predictions, ignore_index=True)

walk_forward_results_df.head()

Running walk-forward backtest for 2011...
Running walk-forward backtest for 2012...
Running walk-forward backtest for 2013...
Running walk-forward backtest for 2014...
Running walk-forward backtest for 2015...
Running walk-forward backtest for 2016...
Running walk-forward backtest for 2017...
Running walk-forward backtest for 2018...
Running walk-forward backtest for 2019...
Running walk-forward backtest for 2020...
Running walk-forward backtest for 2021...
Running walk-forward backtest for 2022...
Running walk-forward backtest for 2023...
Running walk-forward backtest for 2024...
Running walk-forward backtest for 2025...


,model,test_season,threshold,total_bets,wins,losses,pushes,win_rate_excluding_pushes,total_profit,total_risked,roi,season_mae,season_rmse,season_r2,num_selected_features
0,Top 20 RF Features + Ridge Regression,2011,0.500,216,111,94,11,0.541,1176.162,21600,0.054,11.163,14.717,0.051,20
1,Top 20 RF Features + Ridge Regression,2011,1.500,171,88,74,9,0.543,973.443,17100,0.057,11.163,14.717,0.051,20
2,Top 20 RF Features + Ridge Regression,2011,2.500,123,56,61,6,0.479,-736.032,12300,-0.060,11.163,14.717,0.051,20
3,Top 20 RF Features + Ridge Regression,2011,3.500,89,43,41,5,0.512,8.822,8900,0.001,11.163,14.717,0.051,20
4,Top 20 RF Features + Ridge Regression,2011,4.500,68,32,32,4,0.500,-117.763,6800,-0.017,11.163,14.717,0.051,20


In [59]:
walk_forward_results_df.sort_values(
    ["threshold", "test_season"]
)

,model,test_season,threshold,total_bets,wins,losses,pushes,win_rate_excluding_pushes,total_profit,total_risked,roi,season_mae,season_rmse,season_r2,num_selected_features
0,Top 20 RF Features + Ridge Regression,2011,0.500,216,111,94,11,0.541,1176.162,21600,0.054,11.163,14.717,0.051,20
5,Top 20 RF Features + Ridge Regression,2012,0.500,211,116,90,5,0.563,2228.279,21100,0.106,11.157,14.288,0.176,20
10,Top 20 RF Features + Ridge Regression,2013,0.500,198,101,94,3,0.518,204.064,19800,0.010,10.123,13.054,0.171,20
15,Top 20 RF Features + Ridge Regression,2014,0.500,206,91,109,6,0.455,-2244.706,20600,-0.109,11.812,14.891,0.110,20
20,Top 20 RF Features + Ridge Regression,2015,0.500,204,96,101,7,0.487,-883.031,20400,-0.043,10.470,13.028,0.157,20
25,Top 20 RF Features + Ridge Regression,2016,0.500,180,92,84,4,0.523,436.186,18000,0.024,9.101,11.702,0.184,20
30,Top 20 RF Features + Ridge Regression,2017,0.500,205,105,94,6,0.528,604.881,20500,0.030,10.135,13.254,0.161,20
35,Top 20 RF Features + Ridge Regression,2018,0.500,170,88,78,4,0.530,698.475,17000,0.041,9.834,12.814,0.183,20
40,Top 20 RF Features + Ridge Regression,2019,0.500,165,79,80,6,0.497,-491.329,16500,-0.030,10.193,13.026,0.198,20
45,Top 20 RF Features + Ridge Regression,2020,0.500,167,82,85,0,0.491,-705.082,16700,-0.042,10.150,12.987,0.183,20


In [60]:
walk_forward_results_df.sort_values(
    "roi",
    ascending=False
).head(20)

,model,test_season,threshold,total_bets,wins,losses,pushes,win_rate_excluding_pushes,total_profit,total_risked,roi,season_mae,season_rmse,season_r2,num_selected_features
74,Top 20 RF Features + Ridge Regression,2025,4.500,1,1,0,0,1.000,95.238,100,0.952,9.987,12.550,0.241,20
69,Top 20 RF Features + Ridge Regression,2024,4.500,1,1,0,0,1.000,90.909,100,0.909,9.623,12.516,0.254,20
67,Top 20 RF Features + Ridge Regression,2024,2.500,9,7,2,0,0.778,438.918,900,0.488,9.623,12.516,0.254,20
68,Top 20 RF Features + Ridge Regression,2024,3.500,4,3,1,0,0.750,177.056,400,0.443,9.623,12.516,0.254,20
73,Top 20 RF Features + Ridge Regression,2025,3.500,8,6,2,0,0.750,344.268,800,0.430,9.987,12.550,0.241,20
58,Top 20 RF Features + Ridge Regression,2022,3.500,10,7,3,0,0.700,374.552,1000,0.375,8.774,11.639,0.112,20
59,Top 20 RF Features + Ridge Regression,2022,4.500,3,2,1,0,0.667,92.308,300,0.308,8.774,11.639,0.112,20
52,Top 20 RF Features + Ridge Regression,2021,2.500,29,19,9,1,0.679,887.918,2900,0.306,10.762,13.645,0.215,20
13,Top 20 RF Features + Ridge Regression,2013,3.500,59,39,20,0,0.661,1712.830,5900,0.290,10.123,13.054,0.171,20
14,Top 20 RF Features + Ridge Regression,2013,4.500,34,22,12,0,0.647,913.656,3400,0.269,10.123,13.054,0.171,20


## Overall Walk-Forward Threshold Performance

Aggregating the 15 out-of-sample test seasons provides a much larger view of the betting strategy:

| Edge threshold | Bets | Win rate* | Profit | ROI |
|---|---:|---:|---:|---:|
| 0.5 | 2,803 | 51.1% | -$1,546.72 | -0.6% |
| 1.5 | 1,472 | 51.2% | -$66.45 | ~0.0% |
| 2.5 | 749 | 52.6% | +$2,077.15 | **2.8%** |
| 3.5 | 402 | 54.2% | +$2,364.81 | **5.9%** |
| 4.5 | 230 | 53.8% | +$1,260.60 | **5.5%** |

*Win rate excludes pushes.

The walk-forward results provide stronger evidence than the two-season holdout because they use hundreds or thousands of bets across many independently tested seasons. Profitability begins at the 2.5-point threshold, while **3.5 points produces the highest overall ROI** among the thresholds tested.

In [61]:
overall_walk_forward_thresholds = (
    walk_forward_results_df
    .groupby("threshold")
    .agg(
        total_bets=("total_bets", "sum"),
        wins=("wins", "sum"),
        losses=("losses", "sum"),
        pushes=("pushes", "sum"),
        total_profit=("total_profit", "sum"),
        total_risked=("total_risked", "sum"),
        avg_season_mae=("season_mae", "mean")
    )
    .reset_index()
)

overall_walk_forward_thresholds["win_rate_excluding_pushes"] = np.where(
    (overall_walk_forward_thresholds["wins"] + overall_walk_forward_thresholds["losses"]) > 0,
    overall_walk_forward_thresholds["wins"] /
    (overall_walk_forward_thresholds["wins"] + overall_walk_forward_thresholds["losses"]),
    np.nan
)

overall_walk_forward_thresholds["roi"] = np.where(
    overall_walk_forward_thresholds["total_risked"] > 0,
    overall_walk_forward_thresholds["total_profit"] /
    overall_walk_forward_thresholds["total_risked"],
    np.nan
)

overall_walk_forward_thresholds

,threshold,total_bets,wins,losses,pushes,total_profit,total_risked,avg_season_mae,win_rate_excluding_pushes,roi
0,0.500,2803,1393,1335,75,-1546.722,280300,10.221,0.511,-0.006
1,1.500,1472,731,697,44,-66.453,147200,10.221,0.512,-0.000
2,2.500,749,382,344,23,2077.148,74900,10.221,0.526,0.028
3,3.500,402,212,179,11,2364.811,40200,10.221,0.542,0.059
4,4.500,230,120,103,7,1260.599,23000,10.221,0.538,0.055


In [62]:
fig = px.bar(
    overall_walk_forward_thresholds,
    x="threshold",
    y="roi",
    title="Overall Walk-Forward ROI by Threshold",
    labels={
        "threshold": "Model Edge Threshold",
        "roi": "ROI"
    }
)

fig.show()

In [63]:
fig = px.bar(
    overall_walk_forward_thresholds,
    x="threshold",
    y="total_profit",
    title="Overall Walk-Forward Profit by Threshold",
    labels={
        "threshold": "Model Edge Threshold",
        "total_profit": "Total Profit ($)"
    }
)

fig.show()

In [64]:
fig = px.bar(
    overall_walk_forward_thresholds,
    x="threshold",
    y="total_bets",
    title="Overall Walk-Forward Number of Bets by Threshold",
    labels={
        "threshold": "Model Edge Threshold",
        "total_bets": "Number of Bets"
    }
)

fig.show()

## Season-by-Season Walk-Forward Performance

Aggregate profitability can hide unstable year-to-year results, so the notebook also plots ROI and profit separately for each test season.

The 3.5-point threshold has the strongest season consistency among the tested levels: it is profitable in **11 of 15 seasons (73.3%)**. The 2.5-point threshold is profitable in **9 of 15 seasons (60.0%)**.

The lower-volume 4.5-point strategy has positive aggregate ROI but is profitable in only **7 of 15 seasons**, showing why both total return and season consistency matter when evaluating a betting rule.

In [65]:
season_walk_forward = (
    walk_forward_results_df
    .groupby(["test_season", "threshold"])
    .agg(
        total_bets=("total_bets", "sum"),
        wins=("wins", "sum"),
        losses=("losses", "sum"),
        pushes=("pushes", "sum"),
        total_profit=("total_profit", "sum"),
        total_risked=("total_risked", "sum"),
        season_mae=("season_mae", "mean"),
        season_rmse=("season_rmse", "mean"),
        season_r2=("season_r2", "mean")
    )
    .reset_index()
)

season_walk_forward["win_rate_excluding_pushes"] = np.where(
    (season_walk_forward["wins"] + season_walk_forward["losses"]) > 0,
    season_walk_forward["wins"] /
    (season_walk_forward["wins"] + season_walk_forward["losses"]),
    np.nan
)

season_walk_forward["roi"] = np.where(
    season_walk_forward["total_risked"] > 0,
    season_walk_forward["total_profit"] / season_walk_forward["total_risked"],
    np.nan
)

season_walk_forward.head()

,test_season,threshold,total_bets,wins,losses,pushes,total_profit,total_risked,season_mae,season_rmse,season_r2,win_rate_excluding_pushes,roi
0,2011,0.500,216,111,94,11,1176.162,21600,11.163,14.717,0.051,0.541,0.054
1,2011,1.500,171,88,74,9,973.443,17100,11.163,14.717,0.051,0.543,0.057
2,2011,2.500,123,56,61,6,-736.032,12300,11.163,14.717,0.051,0.479,-0.060
3,2011,3.500,89,43,41,5,8.822,8900,11.163,14.717,0.051,0.512,0.001
4,2011,4.500,68,32,32,4,-117.763,6800,11.163,14.717,0.051,0.500,-0.017


In [66]:
fig = px.line(
    season_walk_forward,
    x="test_season",
    y="roi",
    color="threshold",
    markers=True,
    title="Walk-Forward ROI by Season and Threshold",
    labels={
        "test_season": "Test Season",
        "roi": "ROI",
        "threshold": "Threshold"
    }
)

fig.show()

In [67]:
fig = px.line(
    season_walk_forward,
    x="test_season",
    y="total_profit",
    color="threshold",
    markers=True,
    title="Walk-Forward Profit by Season and Threshold",
    labels={
        "test_season": "Test Season",
        "total_profit": "Total Profit ($)",
        "threshold": "Threshold"
    }
)

fig.show()

In [68]:
season_prediction_performance = (
    walk_forward_predictions_df
    .groupby("test_season")
    .agg(
        games=("game_id", "count"),
        mae=("season_mae", "mean"),
        rmse=("season_rmse", "mean"),
        r2=("season_r2", "mean")
    )
    .reset_index()
)

season_prediction_performance

,test_season,games,mae,rmse,r2
0,2011,240,11.163,14.717,0.051
1,2012,240,11.157,14.288,0.176
2,2013,240,10.123,13.054,0.171
3,2014,240,11.812,14.891,0.110
4,2015,240,10.470,13.028,0.157
5,2016,240,9.101,11.702,0.184
6,2017,241,10.135,13.254,0.161
7,2018,240,9.834,12.814,0.183
8,2019,240,10.193,13.026,0.198
9,2020,240,10.150,12.987,0.183


In [69]:
fig = px.line(
    season_prediction_performance,
    x="test_season",
    y="mae",
    markers=True,
    title="Walk-Forward Margin Prediction MAE by Season",
    labels={
        "test_season": "Test Season",
        "mae": "MAE"
    }
)

fig.show()

## Walk-Forward Results Interpretation

The walk-forward analysis is the strongest historical performance test in this notebook because every season is predicted using only information from earlier seasons.

Key findings:

- Average season MAE is approximately **10.22 points**.
- Season MAE ranges from **8.774 in 2022** to **11.812 in 2014**.
- The most recent seasons are relatively strong for margin prediction: **9.623 MAE in 2024** and **9.987 in 2025**.
- Betting every small disagreement is not profitable: the 0.5-point threshold loses **0.6% ROI**, and 1.5 points is essentially break-even.
- Selectivity improves the historical betting results: **2.5 points returns 2.8% ROI**, **3.5 points returns 5.9%**, and **4.5 points returns 5.5%**.
- The 3.5-point threshold combines the highest aggregate ROI with the best season consistency, showing a positive return in **11 of 15 test seasons**.

These results support the project's main betting conclusion: the model is more useful as a selective filter than as a reason to bet every game. The sportsbook remains difficult to beat consistently, and the edge threshold materially changes both the number and quality of bets.

## Scenario Performance Analysis

This section asks **where** the walk-forward model performs best when the main **2.5-point edge threshold** is applied.

The analysis uses the same honest walk-forward predictions described above, so each game was generated from a model trained only on prior seasons. It compares:

- division vs. non-division games,
- home favorites vs. away favorites,
- sportsbook spread ranges,
- game-total ranges,
- season segments,
- home-side vs. away-side bets,
- model-edge ranges.

For each situation, the notebook reports both prediction error and betting results. The ranking requires at least **25 bets across at least 3 seasons** so tiny one-off categories do not automatically rise to the top.

These categories can overlap. For example, one game may simultaneously be an away favorite, a non-division game, a Week 11–14 matchup, and a 3.5–4.9-point model edge. The scenario table should therefore be read as separate diagnostic views, not as mutually exclusive portfolios.

In [70]:
# Use the same threshold selected earlier in the notebook.
scenario_threshold = selected_threshold

scenario_results = walk_forward_predictions_df.copy()

# Add scenario columns from the cleaned games table if they are not
# already present in the walk-forward prediction dataset.
scenario_source_cols = [
    "game_id",
    "div_game",
    "total_line",
]

scenario_cols_to_merge = [
    col for col in scenario_source_cols
    if col != "game_id"
    and col in games_cleaned.columns
    and col not in scenario_results.columns
]

if scenario_cols_to_merge:
    scenario_results = scenario_results.merge(
        games_cleaned[
            ["game_id"] + scenario_cols_to_merge
        ].drop_duplicates("game_id"),
        on="game_id",
        how="left",
        validate="many_to_one"
    )

# Run the existing betting logic on all honest walk-forward predictions.
scenario_results = backtest_spread_bets(
    scenario_results,
    threshold=scenario_threshold,
    stake=100
)

def division_label(value):
    if pd.isna(value):
        return "Unknown"

    text = str(value).strip().lower()

    if text in {"1", "true", "yes", "y", "division", "divisional"}:
        return "Division"
    if text in {"0", "false", "no", "n", "non-division", "non_division"}:
        return "Non-Division"

    return "Unknown"

if "div_game" in scenario_results.columns:
    scenario_results["division_status"] = (
        scenario_results["div_game"].apply(division_label)
    )
else:
    scenario_results["division_status"] = "Unknown"

scenario_results["favorite_type"] = np.select(
    [
        scenario_results["market_home_margin"] > 0,
        scenario_results["market_home_margin"] < 0
    ],
    [
        "Home Favorite",
        "Away Favorite"
    ],
    default="Pick'em"
)

scenario_results["market_spread_range"] = pd.cut(
    scenario_results["market_home_margin"].abs(),
    bins=[-np.inf, 2.5, 6.5, 9.5, np.inf],
    labels=[
        "0-2.5 Points",
        "3-6.5 Points",
        "7-9.5 Points",
        "10+ Points"
    ]
).astype(str)

if "total_line" in scenario_results.columns:
    total_numeric = pd.to_numeric(
        scenario_results["total_line"],
        errors="coerce"
    )

    scenario_results["game_total_range"] = pd.cut(
        total_numeric,
        bins=[-np.inf, 41, 46, 50, np.inf],
        labels=[
            "41 or Lower",
            "41.5-46",
            "46.5-50",
            "Over 50"
        ]
    ).astype(str)

    scenario_results.loc[
        total_numeric.isna(),
        "game_total_range"
    ] = "Unknown"
else:
    scenario_results["game_total_range"] = "Unknown"

scenario_results["season_segment"] = scenario_results["week"].apply(
    assign_season_segment
)

absolute_edge = scenario_results["model_edge"].abs()

scenario_results["model_edge_range"] = np.select(
    [
        absolute_edge < scenario_threshold,
        absolute_edge < 3.5,
        absolute_edge < 5.0,
        absolute_edge < 7.5
    ],
    [
        "Below Bet Threshold",
        f"{scenario_threshold}-3.4 Points",
        "3.5-4.9 Points",
        "5.0-7.4 Points"
    ],
    default="7.5+ Points"
)

print("Scenario threshold:", scenario_threshold)
print("Walk-forward games analyzed:", len(scenario_results))
print(
    "Actual bets analyzed:",
    scenario_results["bet_side"].isin(["home", "away"]).sum()
)

Scenario threshold: 2.5
Walk-forward games analyzed: 3680
Actual bets analyzed: 749


In [71]:
def summarize_scenario_group(group):
    # Summarize prediction and betting performance for one scenario.
    bets = group[
        group["bet_side"].isin(["home", "away"])
    ].copy()

    total_bets = len(bets)
    wins = (bets["bet_result"] == "win").sum()
    losses = (bets["bet_result"] == "loss").sum()
    pushes = (bets["bet_result"] == "push").sum()

    decisions = wins + losses
    win_rate = (
        wins / decisions
        if decisions > 0
        else np.nan
    )

    total_profit = bets["profit"].sum()
    total_risked = total_bets * 100
    roi = (
        total_profit / total_risked
        if total_risked > 0
        else np.nan
    )

    if total_bets > 0:
        season_profit = bets.groupby("test_season")["profit"].sum()
        seasons_with_bets = season_profit.index.nunique()
        profitable_seasons = (season_profit > 0).sum()
        profitable_season_rate = (
            profitable_seasons / seasons_with_bets
            if seasons_with_bets > 0
            else np.nan
        )
    else:
        seasons_with_bets = 0
        profitable_seasons = 0
        profitable_season_rate = np.nan

    model_mae = mean_absolute_error(
        group["home_margin"],
        group["predicted_home_margin"]
    )
    market_mae = mean_absolute_error(
        group["home_margin"],
        group["market_home_margin"]
    )

    return {
        "games": len(group),
        "model_mae": model_mae,
        "sportsbook_mae": market_mae,
        "mae_improvement_vs_sportsbook": (
            market_mae - model_mae
        ),
        "total_bets": total_bets,
        "wins": wins,
        "losses": losses,
        "pushes": pushes,
        "win_rate_excluding_pushes": win_rate,
        "total_profit": total_profit,
        "roi": roi,
        "seasons_with_bets": seasons_with_bets,
        "profitable_seasons": profitable_seasons,
        "profitable_season_rate": profitable_season_rate
    }


scenario_dimensions = {
    "Division Status": "division_status",
    "Favorite Type": "favorite_type",
    "Market Spread Range": "market_spread_range",
    "Game Total Range": "game_total_range",
    "Season Segment": "season_segment",
    "Bet Side": "bet_side",
    "Model Edge Range": "model_edge_range"
}

scenario_summary_rows = []

for scenario_name, scenario_column in scenario_dimensions.items():
    for scenario_value, group in scenario_results.groupby(
        scenario_column,
        dropna=False
    ):
        summary = summarize_scenario_group(group)

        scenario_summary_rows.append({
            "scenario": scenario_name,
            "situation": scenario_value,
            **summary
        })

scenario_performance = pd.DataFrame(
    scenario_summary_rows
).sort_values(
    ["scenario", "roi"],
    ascending=[True, False]
).reset_index(drop=True)

scenario_performance

,scenario,situation,games,model_mae,sportsbook_mae,mae_improvement_vs_sportsbook,total_bets,wins,losses,pushes,win_rate_excluding_pushes,total_profit,roi,seasons_with_bets,profitable_seasons,profitable_season_rate
0,Bet Side,home,336,9.956,9.408,-0.549,336,171,153,12,0.528,1059.493,0.032,15,7,0.467
1,Bet Side,away,413,10.789,10.103,-0.686,413,211,191,11,0.525,1017.654,0.025,15,10,0.667
2,Bet Side,no_bet,2931,10.161,10.101,-0.060,0,0,0,0,NaN,0.000,NaN,0,0,NaN
3,Division Status,Non-Division,2324,10.380,10.231,-0.149,492,255,226,11,0.530,1778.045,0.036,15,8,0.533
4,Division Status,Division,1356,9.926,9.708,-0.218,257,127,118,12,0.518,299.103,0.012,15,6,0.400
5,Favorite Type,Away Favorite,1380,10.131,10.033,-0.098,266,145,110,11,0.569,2771.634,0.104,15,11,0.733
6,Favorite Type,Home Favorite,2296,10.259,10.039,-0.221,481,236,233,12,0.503,-685.396,-0.014,15,7,0.467
7,Favorite Type,Pick'em,4,11.705,11.500,-0.205,2,1,1,0,0.500,-9.091,-0.045,1,0,0.000
8,Game Total Range,46.5-50,974,10.124,9.971,-0.153,188,97,84,7,0.536,968.305,0.052,15,8,0.533
9,Game Total Range,41.5-46,1558,10.258,10.133,-0.125,313,161,144,8,0.528,1057.531,0.034,15,8,0.533


In [72]:
# Rank only situations with a reasonable sample across multiple seasons.
minimum_bets = 25
minimum_seasons = 3

best_model_spots = (
    scenario_performance[
        (scenario_performance["total_bets"] >= minimum_bets)
        & (
            scenario_performance["seasons_with_bets"]
            >= minimum_seasons
        )
        & (
            scenario_performance["situation"]
            != "Below Bet Threshold"
        )
        & (
            scenario_performance["situation"]
            != "no_bet"
        )
        & (
            scenario_performance["situation"]
            != "no_odds"
        )
        & (
            scenario_performance["situation"]
            != "Unknown"
        )
    ]
    .sort_values(
        [
            "roi",
            "profitable_season_rate",
            "win_rate_excluding_pushes"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

display_columns = [
    "scenario",
    "situation",
    "games",
    "model_mae",
    "sportsbook_mae",
    "mae_improvement_vs_sportsbook",
    "total_bets",
    "wins",
    "losses",
    "win_rate_excluding_pushes",
    "total_profit",
    "roi",
    "seasons_with_bets",
    "profitable_seasons",
    "profitable_season_rate"
]

print("Best model situations with sufficient sample size")
display(best_model_spots[display_columns].head(20))

positive_stable_spots = best_model_spots[
    (best_model_spots["roi"] > 0)
    & (
        best_model_spots["profitable_season_rate"]
        >= 0.50
    )
].copy()

print("\nStable positive situations")
if positive_stable_spots.empty:
    print(
        "No tested situation had both positive overall ROI and "
        "profitable results in at least half of its seasons."
    )
else:
    display(
        positive_stable_spots[display_columns].head(10)
    )

Best model situations with sufficient sample size


,scenario,situation,games,model_mae,sportsbook_mae,mae_improvement_vs_sportsbook,total_bets,wins,losses,win_rate_excluding_pushes,total_profit,roi,seasons_with_bets,profitable_seasons,profitable_season_rate
0,Favorite Type,Away Favorite,1380,10.131,10.033,-0.098,266,145,110,0.569,2771.634,0.104,15,11,0.733
1,Model Edge Range,3.5-4.9 Points,223,10.119,9.787,-0.332,223,120,98,0.550,1633.524,0.073,15,9,0.600
2,Season Segment,Weeks 11-14,901,9.742,9.588,-0.154,120,64,52,0.552,850.006,0.071,13,8,0.615
3,Season Segment,Weeks 15-18,799,10.316,10.204,-0.112,99,53,44,0.546,665.953,0.067,11,5,0.455
4,Market Spread Range,10+ Points,478,10.311,10.146,-0.165,92,48,41,0.539,522.139,0.057,12,6,0.500
5,Market Spread Range,3-6.5 Points,1751,10.130,9.991,-0.139,358,188,160,0.540,2013.601,0.056,15,9,0.600
6,Game Total Range,46.5-50,974,10.124,9.971,-0.153,188,97,84,0.536,968.305,0.052,15,8,0.533
7,Model Edge Range,5.0-7.4 Points,124,10.759,10.085,-0.675,124,65,56,0.537,590.440,0.048,15,7,0.467
8,Season Segment,Weeks 2-5,926,10.285,10.001,-0.284,338,174,154,0.530,1225.410,0.036,15,10,0.667
9,Division Status,Non-Division,2324,10.380,10.231,-0.149,492,255,226,0.530,1778.045,0.036,15,8,0.533



Stable positive situations


,scenario,situation,games,model_mae,sportsbook_mae,mae_improvement_vs_sportsbook,total_bets,wins,losses,win_rate_excluding_pushes,total_profit,roi,seasons_with_bets,profitable_seasons,profitable_season_rate
0,Favorite Type,Away Favorite,1380,10.131,10.033,-0.098,266,145,110,0.569,2771.634,0.104,15,11,0.733
1,Model Edge Range,3.5-4.9 Points,223,10.119,9.787,-0.332,223,120,98,0.550,1633.524,0.073,15,9,0.600
2,Season Segment,Weeks 11-14,901,9.742,9.588,-0.154,120,64,52,0.552,850.006,0.071,13,8,0.615
4,Market Spread Range,10+ Points,478,10.311,10.146,-0.165,92,48,41,0.539,522.139,0.057,12,6,0.500
5,Market Spread Range,3-6.5 Points,1751,10.130,9.991,-0.139,358,188,160,0.540,2013.601,0.056,15,9,0.600
6,Game Total Range,46.5-50,974,10.124,9.971,-0.153,188,97,84,0.536,968.305,0.052,15,8,0.533
8,Season Segment,Weeks 2-5,926,10.285,10.001,-0.284,338,174,154,0.530,1225.410,0.036,15,10,0.667
9,Division Status,Non-Division,2324,10.380,10.231,-0.149,492,255,226,0.530,1778.045,0.036,15,8,0.533
10,Game Total Range,41.5-46,1558,10.258,10.133,-0.125,313,161,144,0.528,1057.531,0.034,15,8,0.533
13,Model Edge Range,7.5+ Points,55,13.494,9.591,-3.903,55,27,25,0.519,140.847,0.026,8,6,0.750


In [73]:
# Simple chart of the strongest sufficiently large situations.
top_scenario_plot = best_model_spots.head(15).copy()

if not top_scenario_plot.empty:
    top_scenario_plot["scenario_label"] = (
        top_scenario_plot["scenario"]
        + ": "
        + top_scenario_plot["situation"].astype(str)
    )

    fig = px.bar(
        top_scenario_plot.sort_values("roi"),
        x="roi",
        y="scenario_label",
        orientation="h",
        title=(
            f"Best Walk-Forward Situations at a "
            f"{scenario_threshold}-Point Edge"
        ),
        labels={
            "roi": "ROI",
            "scenario_label": "Situation"
        }
    )
    fig.show()

### Scenario Results Interpretation

At the 2.5-point walk-forward threshold, **749 bets** are available across the 15 test seasons. Several situations stand out:

- **Away favorites:** 266 bets, **56.9% win rate**, **10.4% ROI**, and profitable in **11 of 15 seasons**. This is the strongest large-sample scenario tested.
- **3.5–4.9-point model edges:** 223 bets, **55.0% win rate** and **7.3% ROI**.
- **Weeks 11–14:** 120 bets, **55.2% win rate** and **7.1% ROI**.
- **10+ point market spreads:** 92 bets and **5.7% ROI**.
- **3–6.5 point market spreads:** 358 bets and **5.6% ROI**.
- **Non-division games:** 492 bets and **3.6% ROI**, compared with **1.2% ROI** in division games.

Several weaker groups also help define where the model is less useful. Home favorites lose **1.4% ROI**, 0–2.5 point market spreads lose **1.2%**, and 7–9.5 point spreads lose **2.1%**.

One important distinction is that the model's **MAE is worse than the sportsbook in every scenario listed**, even when the bets are profitable. For example, away-favorite games have a 10.131 model MAE versus a 10.033 sportsbook MAE. This reinforces the project's central idea: the model can potentially add value through **selective directional disagreements** without being the better overall margin predictor within that entire category.

## Final Model Performance by Year

The final section refits the **existing final Ridge pipeline** on the entire 2010–2025 modeling dataset and applies it back to every season at the same 2.5-point betting threshold.

Because this pipeline uses `selected_feature_cols`, it reflects the **five-feature final implementation** from the earlier holdout section, not the top-20 walk-forward feature-selection process.

This test answers a descriptive question: *How does the final fitted model look when applied across each historical year?*

It is intentionally **retrospective and in sample** because the season being evaluated was also used to fit the model. These values can show how the final fitted relationships behave across eras, but they must not be presented as out-of-sample evidence. The walk-forward results remain the appropriate historical performance test.

In [74]:
from sklearn.base import clone

# Refit the exact existing final model on all available seasons.
historical_final_model = clone(final_model)

historical_final_model.fit(
    model_df[selected_feature_cols],
    model_df[target_col]
)

historical_predictions = historical_final_model.predict(
    model_df[selected_feature_cols]
)

historical_final_results = model_df.copy()
historical_final_results["predicted_home_margin"] = (
    historical_predictions
)
historical_final_results["model_edge"] = (
    historical_final_results["predicted_home_margin"]
    - historical_final_results["market_home_margin"]
)

historical_final_backtest = backtest_spread_bets(
    historical_final_results,
    threshold=scenario_threshold,
    stake=100
)

final_model_year_rows = []

for season, group in historical_final_backtest.groupby("season"):
    bets = group[
        group["bet_side"].isin(["home", "away"])
    ].copy()

    wins = (bets["bet_result"] == "win").sum()
    losses = (bets["bet_result"] == "loss").sum()
    pushes = (bets["bet_result"] == "push").sum()
    decisions = wins + losses

    total_bets = len(bets)
    total_profit = bets["profit"].sum()
    total_risked = total_bets * 100

    model_mae = mean_absolute_error(
        group["home_margin"],
        group["predicted_home_margin"]
    )
    sportsbook_mae = mean_absolute_error(
        group["home_margin"],
        group["market_home_margin"]
    )

    final_model_year_rows.append({
        "season": int(season),
        "games": len(group),
        "model_mae": model_mae,
        "sportsbook_mae": sportsbook_mae,
        "mae_improvement_vs_sportsbook": (
            sportsbook_mae - model_mae
        ),
        "total_bets": total_bets,
        "wins": wins,
        "losses": losses,
        "pushes": pushes,
        "win_rate_excluding_pushes": (
            wins / decisions
            if decisions > 0
            else np.nan
        ),
        "total_profit": total_profit,
        "roi": (
            total_profit / total_risked
            if total_risked > 0
            else np.nan
        )
    })

final_model_yearly_performance = (
    pd.DataFrame(final_model_year_rows)
    .sort_values("season")
    .reset_index(drop=True)
)

final_model_yearly_performance

,season,games,model_mae,sportsbook_mae,mae_improvement_vs_sportsbook,total_bets,wins,losses,pushes,win_rate_excluding_pushes,total_profit,roi
0,2010,240,11.032,11.065,0.032,7,5,2,0,0.714,289.146,0.413
1,2011,240,10.488,10.369,-0.119,4,1,2,1,0.333,-103.846,-0.260
2,2012,240,10.848,10.854,0.006,8,4,3,1,0.571,95.581,0.119
3,2013,240,10.031,10.015,-0.017,8,3,5,0,0.375,-186.000,-0.233
4,2014,240,11.453,11.477,0.024,9,6,3,0,0.667,267.559,0.297
5,2015,240,10.198,10.177,-0.021,9,4,5,0,0.444,-111.718,-0.124
6,2016,240,8.976,9.094,0.118,9,7,2,0,0.778,480.641,0.534
7,2017,241,9.807,9.925,0.119,8,5,3,0,0.625,175.396,0.219
8,2018,240,9.827,9.881,0.055,3,3,0,0,1.000,297.336,0.991
9,2019,240,10.065,10.065,-0.000,8,3,5,0,0.375,-200.910,-0.251


In [75]:
fig = px.bar(
    final_model_yearly_performance,
    x="season",
    y="roi",
    title=(
        f"Final Fitted Model ROI by Historical Season "
        f"at a {scenario_threshold}-Point Edge"
    ),
    labels={
        "season": "Season",
        "roi": "ROI"
    }
)
fig.show()

# Save the two requested result tables in the existing models folder.
scenario_performance.to_csv(
    models_dir / "scenario_performance.csv",
    index=False
)

final_model_yearly_performance.to_csv(
    models_dir / "final_model_yearly_performance.csv",
    index=False
)

print("Saved:")
print(models_dir / "scenario_performance.csv")
print(models_dir / "final_model_yearly_performance.csv")

Saved:
/content/drive/MyDrive/DATA 606/nfl_spread_capstone/models/scenario_performance.csv
/content/drive/MyDrive/DATA 606/nfl_spread_capstone/models/final_model_yearly_performance.csv


### Retrospective Year-by-Year Takeaway

The full-fit five-feature model produces **106 total 2.5-point bets** when applied retrospectively across 2010–2025. It records 60 wins, 41 losses, and 5 pushes for approximately **+$1,699 in profit and 16.0% ROI**, with positive profit in **11 of 16 seasons**.

Prediction accuracy is much less dramatic. The model's average yearly MAE is about **10.095**, compared with **10.108** for the sportsbook, and it beats the sportsbook's MAE in **8 of 16 seasons**. The average margin-accuracy advantage is only about **0.013 points**.

These retrospective betting numbers should not be compared directly with the walk-forward ROI as if they were equally valid tests. The final model was trained on the same seasons it is evaluating, so the backcast is expected to look more favorable. The **walk-forward 2.5-point result of 2.8% ROI across 749 bets** is the more defensible estimate of historical out-of-sample performance.